# Kuantum Makine Öğrenmesi ile Obezite Sınıflandırması

Bu defter, UCI Obezite veri seti (Palechor & De la Hoz Manotas, 2019) üzerinde
klasik makine öğrenmesi modelleri ile kuantum makine öğrenmesi (QML) modellerini
karşılaştırmalı olarak değerlendirir.

## İçerik

1. Veri yükleme ve keşifsel analiz (EDA)
2. Klasik ve kuantum ön işleme
3. Altı klasik model (LR, SVM-RBF, RF, XGBoost, KNN, MLP) — 16 ve top-10 özellik
4. Yedi kuantum model (Angle, Amplitude, Re-uploading ablation, dual-branch hibrit)
5. Gürültü dayanıklılık analizi
6. SHAP yorumlanabilirlik analizi
7. Eşli Wilcoxon işaretli sıra testi
8. Final konsolidasyon

## Yeniden üretilebilirlik

- Random state: 42
- Çapraz doğrulama: 5-fold StratifiedKFold
- Kuantum framework: PennyLane + PyTorch (`default.qubit`, `backprop`)
- Tüm çıktılar `./output` altına yazılır


In [ ]:
import os

OUTPUT_DIR = './output'

for klasor in [
    OUTPUT_DIR,
    f'{OUTPUT_DIR}/sonuclar',
    f'{OUTPUT_DIR}/sonuclar/quantum',
    f'{OUTPUT_DIR}/sonuclar/klasik',
    f'{OUTPUT_DIR}/gorseller',
    f'{OUTPUT_DIR}/modeller',
    f'{OUTPUT_DIR}/checkpoints',
    f'{OUTPUT_DIR}/veri',
]:
    os.makedirs(klasor, exist_ok=True)


In [ ]:
# Yerel ortam için requirements.txt'i kullanın.
# Notebook ortamında çalıştırıyorsanız aşağıdaki satırı açın:
# !pip install -q pennylane pennylane-lightning xgboost shap scipy ucimlrepo


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import time
import json
import pickle
from datetime import datetime

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_val_score,
)
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    LabelEncoder,
    OneHotEncoder,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    matthews_corrcoef,
    cohen_kappa_score,
    ConfusionMatrixDisplay,
)

from scipy.stats import wilcoxon
from scipy import stats

import pennylane as qml
from pennylane import numpy as pnp
import torch
import torch.nn as nn
import torch.optim as optim

import shap

# Sürüm bilgisi (yeniden üretilebilirlik)
versiyonlar = {
    "NumPy"       : np.__version__,
    "Pandas"      : pd.__version__,
    "PennyLane"   : qml.__version__,
    "PyTorch"     : torch.__version__,
    "Scikit-learn": __import__('sklearn').__version__,
    "XGBoost"     : xgb.__version__,
    "SciPy"       : __import__('scipy').__version__,
    "SHAP"        : shap.__version__,
    "Matplotlib"  : matplotlib.__version__,
}
for k, v in versiyonlar.items():
    print(f"{k:<14}: {v}")

# Çıktı yolları
SONUC_QUANTUM = f'{OUTPUT_DIR}/sonuclar/quantum'
SONUC_KLASIK  = f'{OUTPUT_DIR}/sonuclar/klasik'
GORSELLER     = f'{OUTPUT_DIR}/gorseller'
MODELLER      = f'{OUTPUT_DIR}/modeller'
CHECKPOINT    = f'{OUTPUT_DIR}/checkpoints'
VERI_DIR      = f'{OUTPUT_DIR}/veri'

# Deney sabitleri
RANDOM_STATE = 42
TEST_SIZE    = 0.20
N_SPLITS     = 5
N_CLASSES    = 7


In [ ]:
# Veri yükleme: yerel yedek varsa onu kullan, yoksa UCI'den indir
yedek_yol = f'{VERI_DIR}/obezite_raw.csv'

if os.path.exists(yedek_yol):
    print(f"Yerel yedekten yükleniyor: {yedek_yol}")
    df = pd.read_csv(yedek_yol)
else:
    print("UCI Repository'den indiriliyor (id=544)...")
    from ucimlrepo import fetch_ucirepo
    veri_uci = fetch_ucirepo(id=544)
    df = pd.concat([veri_uci.data.features, veri_uci.data.targets], axis=1)
    df = df.rename(columns={df.columns[-1]: 'NObeyesdad'})
    df.to_csv(yedek_yol, index=False)
    print(f"Yerel yedek kaydedildi: {yedek_yol}")

print(f"\nVeri yüklendi: {df.shape[0]} satır, {df.shape[1]} sütun")

# Duplicate temizliği
n_once = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
n_sonra = df.shape[0]
print(f"Duplicate temizliği: {n_once} → {n_sonra} (silinen: {n_once - n_sonra})")

# Sınıf dağılımı
print(f"\nSınıf dağılımı (NObeyesdad):")
sinif_dagilim = df['NObeyesdad'].value_counts()
sinif_yuzde = df['NObeyesdad'].value_counts(normalize=True) * 100
for sinif in sinif_dagilim.index:
    print(f"  {sinif:<25}: {sinif_dagilim[sinif]:>4} (%{sinif_yuzde[sinif]:.1f})")

# Özellik türleri
sayisal_sutunlar = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
kategorik_sutunlar = df.select_dtypes(include=['object']).columns.tolist()
kategorik_sutunlar = [c for c in kategorik_sutunlar if c != 'NObeyesdad']

print(f"\nÖzellik türleri:")
print(f"  Sayısal   ({len(sayisal_sutunlar)}): {sayisal_sutunlar}")
print(f"  Kategorik ({len(kategorik_sutunlar)}): {kategorik_sutunlar}")

# Eksik veri kontrolü
eksikler = df.isnull().sum().sum()
print(f"\nToplam eksik değer: {eksikler}")

# Sınıf dağılımı grafiği
plt.figure(figsize=(11, 5))
sinif_sirali = df['NObeyesdad'].value_counts().index
sns.countplot(data=df, x='NObeyesdad', order=sinif_sirali, palette='viridis')
plt.xticks(rotation=30, ha='right')
plt.title('Obezite Veri Seti - Sınıf Dağılımı', fontweight='bold')
plt.xlabel('Sınıf')
plt.ylabel('Örnek Sayısı')
plt.tight_layout()
plt.savefig(f'{GORSELLER}/sinif_dagilimi.png', dpi=150, bbox_inches='tight')
plt.show()

# EDA raporunu kaydet
eda_rapor = {
    "tarih"            : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "veri_seti"        : "UCI Obesity (Palechor & De la Hoz Manotas, 2019)",
    "n_ornek_ham"      : int(n_once),
    "n_ornek_temiz"    : int(n_sonra),
    "n_ozellik"        : int(df.shape[1] - 1),
    "n_sinif"          : int(df['NObeyesdad'].nunique()),
    "sayisal_ozellik"  : sayisal_sutunlar,
    "kategorik_ozellik": kategorik_sutunlar,
    "sinif_dagilim"    : {k: int(v) for k, v in sinif_dagilim.items()},
    "sinif_yuzde"      : {k: float(v) for k, v in sinif_yuzde.items()},
    "eksik_veri"       : int(eksikler),
}

with open(f'{OUTPUT_DIR}/sonuclar/eda_rapor.json', 'w', encoding='utf-8') as f:
    json.dump(eda_rapor, f, ensure_ascii=False, indent=4)


In [ ]:
print("=" * 60)
print("⚙️  KLASİK PREPROCESSING")
print("=" * 60)

# ── Adım 1: Hedef ve Özellik Ayrımı ─────────────────────────
target_col = 'NObeyesdad'
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"\n📌 X şekli: {X.shape}")
print(f"📌 y şekli: {y.shape}")

# ── Adım 2: Label Encoding (multiclass için) ────────────────
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"\n📌 SINIF KODLAMASI:")
for i, sinif in enumerate(label_encoder.classes_):
    print(f"   {i} → {sinif}")

# ── Adım 3: Train/Test Split (80/20, stratified) ────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y_encoded
)

print(f"\n📌 TRAIN/TEST AYRIMI:")
print(f"   Eğitim seti : {X_train.shape[0]} örnek "
      f"({X_train.shape[0]/len(y)*100:.1f}%)")
print(f"   Test seti   : {X_test.shape[0]} örnek "
      f"({X_test.shape[0]/len(y)*100:.1f}%)")

# Sınıf dağılım kontrolü (split sonrası)
print(f"\n   Train sınıf dağılımı:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    sinif_adi = label_encoder.classes_[u]
    print(f"     {sinif_adi:<25}: {c}")

# ── Adım 4: Sayısal vs Kategorik Özellikler ─────────────────
sayisal_ozellikler = ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
kategorik_ozellikler = ['Gender', 'family_history_with_overweight', 'FAVC',
                         'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']

print(f"\n📌 ÖZELLİK GRUPLARI:")
print(f"   Sayısal ({len(sayisal_ozellikler)})   : {sayisal_ozellikler}")
print(f"   Kategorik ({len(kategorik_ozellikler)}) : {kategorik_ozellikler}")

# ── Adım 5: ColumnTransformer (Klasik 16-Özellik) ───────────
preprocessor_16 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), sayisal_ozellikler),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         kategorik_ozellikler)
    ]
)

print(f"\n📌 PREPROCESSOR (16 özellik): StandardScaler + OneHotEncoder")

# ── Adım 6: Top-10 Feature Listesi ──────────────────────────
# Önceki RF importance analizinden seçilen klinik anlamlı özellikler
top10_features = [
    'Weight',
    'Age',
    'FCVC',
    'Height',
    'Gender',
    'NCP',
    'CALC',
    'FAF',
    'TUE',
    'family_history_with_overweight'
]

# Top-10 sayısal/kategorik ayrımı
top10_sayisal = [f for f in top10_features if f in sayisal_ozellikler]
top10_kategorik = [f for f in top10_features if f in kategorik_ozellikler]

print(f"\n📌 TOP-10 ÖZELLİKLER (RF Importance ile seçilmiş):")
print(f"   Sayısal ({len(top10_sayisal)})   : {top10_sayisal}")
print(f"   Kategorik ({len(top10_kategorik)}) : {top10_kategorik}")

# ── Adım 7: Top-10 Train/Test Versiyonları ──────────────────
X_train_10 = X_train[top10_features].copy()
X_test_10  = X_test[top10_features].copy()

preprocessor_10 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), top10_sayisal),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         top10_kategorik)
    ]
)

print(f"\n📌 TOP-10 VERİ ŞEKLİ:")
print(f"   X_train_10: {X_train_10.shape}")
print(f"   X_test_10 : {X_test_10.shape}")

# ── Adım 8: 5-Fold CV Splitter ──────────────────────────────
skf = StratifiedKFold(
    n_splits     = N_SPLITS,
    shuffle      = True,
    random_state = RANDOM_STATE
)

print(f"\n📌 ÇAPRAZ DOĞRULAMA:")
print(f"   Yöntem  : Stratified K-Fold")
print(f"   K değeri: {N_SPLITS}")

fold_boyutlari = []
for fold_no, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    fold_boyutlari.append(len(val_idx))
    print(f"   Fold {fold_no}: Eğitim={len(train_idx)} | Doğrulama={len(val_idx)}")

# ── Adım 9: Tüm Klasik Verileri Sözlüğe Topla ───────────────
KLASIK_VERILER = {
    # 16 özellik
    'X_train_16'        : X_train,
    'X_test_16'         : X_test,
    'preprocessor_16'   : preprocessor_16,

    # 10 özellik (top RF importance)
    'X_train_10'        : X_train_10,
    'X_test_10'         : X_test_10,
    'preprocessor_10'   : preprocessor_10,

    # Etiketler
    'y_train'           : y_train,
    'y_test'            : y_test,
    'label_encoder'     : label_encoder,

    # Özellik listeleri
    'sayisal'           : sayisal_ozellikler,
    'kategorik'         : kategorik_ozellikler,
    'top10_features'    : top10_features,
    'top10_sayisal'     : top10_sayisal,
    'top10_kategorik'   : top10_kategorik,

    # CV splitter
    'skf'               : skf,
}

print(f"\n💾 KLASIK_VERILER sözlüğü oluşturuldu (10 anahtar)")

# ── Preprocessing Raporunu Kaydet ───────────────────────────
prep_rapor = {
    "tarih"             : datetime.now().strftime('%d/%m/%Y %H:%M:%S'),
    "n_train"           : int(X_train.shape[0]),
    "n_test"            : int(X_test.shape[0]),
    "n_ozellik_full"    : 16,
    "n_ozellik_top10"   : 10,
    "n_sinif"           : N_CLASSES,
    "scaling"           : "StandardScaler (sayısal)",
    "encoding"          : "OneHotEncoder (kategorik)",
    "split_strategy"    : f"80/20 stratified, random_state={RANDOM_STATE}",
    "cv_strategy"       : f"{N_SPLITS}-fold StratifiedKFold",
    "top10_features"    : top10_features,
    "label_mapping"     : {int(i): str(c) for i, c in enumerate(label_encoder.classes_)}
}

rapor_yol = f'{OUTPUT_DIR}/sonuclar/preprocessing_klasik_rapor.json'
with open(rapor_yol, 'w', encoding='utf-8') as f:
    json.dump(prep_rapor, f, ensure_ascii=False, indent=4)

print(f"💾 Rapor kaydedildi: {rapor_yol}")


In [ ]:
print("=" * 60)
print("⚛️  QUANTUM PREPROCESSING")
print("=" * 60)

# Top-10 verilerini al
X_train_10 = KLASIK_VERILER['X_train_10']
X_test_10  = KLASIK_VERILER['X_test_10']
y_train    = KLASIK_VERILER['y_train']
y_test     = KLASIK_VERILER['y_test']

# ── Adım 1: Sayısal Özellikleri Ölçekle ─────────────────────
print(f"\n📌 ADIM 1: Sayısal Özellik Ölçekleme")

sayisal_top10 = KLASIK_VERILER['top10_sayisal']
kategorik_top10 = KLASIK_VERILER['top10_kategorik']

# StandardScaler (z-score) - sayısal sütunlar
sayisal_scaler = StandardScaler()
X_train_sayisal_scaled = sayisal_scaler.fit_transform(X_train_10[sayisal_top10])
X_test_sayisal_scaled  = sayisal_scaler.transform(X_test_10[sayisal_top10])

print(f"   Sayısal scaled train shape: {X_train_sayisal_scaled.shape}")
print(f"   Sayısal scaled mean: {X_train_sayisal_scaled.mean():.4f} (≈0)")
print(f"   Sayısal scaled std : {X_train_sayisal_scaled.std():.4f} (≈1)")

# ── Adım 2: Kategorik Özellikleri Ordinal Encode ────────────
print(f"\n📌 ADIM 2: Kategorik Özellik Encoding")

# Manuel ordinal mapping
gender_map = {'Male': 1, 'Female': 0}
calc_map = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
family_map = {'yes': 1, 'no': 0}

X_train_kat = pd.DataFrame()
X_test_kat = pd.DataFrame()

X_train_kat['Gender'] = X_train_10['Gender'].map(gender_map)
X_test_kat['Gender']  = X_test_10['Gender'].map(gender_map)

X_train_kat['CALC'] = X_train_10['CALC'].map(calc_map)
X_test_kat['CALC']  = X_test_10['CALC'].map(calc_map)

X_train_kat['family_history_with_overweight'] = X_train_10['family_history_with_overweight'].map(family_map)
X_test_kat['family_history_with_overweight']  = X_test_10['family_history_with_overweight'].map(family_map)

print(f"   Gender mapping     : {gender_map}")
print(f"   CALC mapping       : {calc_map}")
print(f"   family_history map : {family_map}")
print(f"   Train kategorik shape: {X_train_kat.shape}")

# ── Adım 3: Sayısal + Kategorik Birleştir (10 özellik) ──────
print(f"\n📌 ADIM 3: Sayısal + Kategorik Birleştirme (10 özellik)")

# Sırayı top-10 listesine göre düzenle
X_train_full = np.zeros((X_train_10.shape[0], 10))
X_test_full  = np.zeros((X_test_10.shape[0], 10))

top10_features = KLASIK_VERILER['top10_features']
for i, feat in enumerate(top10_features):
    if feat in sayisal_top10:
        idx = sayisal_top10.index(feat)
        X_train_full[:, i] = X_train_sayisal_scaled[:, idx]
        X_test_full[:, i]  = X_test_sayisal_scaled[:, idx]
    else:
        X_train_full[:, i] = X_train_kat[feat].values
        X_test_full[:, i]  = X_test_kat[feat].values

print(f"   Full 10-feature train: {X_train_full.shape}")
print(f"   Full 10-feature test : {X_test_full.shape}")

# ── Adım 4: Angle Encoding Hazırlığı (clip[-3,3] * π/3) ────
print(f"\n📌 ADIM 4: Angle Encoding Hazırlığı")

X_train_angle = np.clip(X_train_full, -3, 3) * (np.pi / 3)
X_test_angle  = np.clip(X_test_full, -3, 3) * (np.pi / 3)

print(f"   Angle train min/max: {X_train_angle.min():.4f} / {X_train_angle.max():.4f}")
print(f"   Angle test  min/max: {X_test_angle.min():.4f} / {X_test_angle.max():.4f}")
print(f"   Angle aralığı: [-π, π] içinde mi? {np.abs(X_train_angle).max() <= np.pi + 0.01}")

# ── Adım 5: PCA-6 Hazırlığı (WBCD ile Simetri için) ─────────
print(f"\n📌 ADIM 5: PCA-6 Hazırlığı (10 → 6 boyut)")

pca_6 = PCA(n_components=6, random_state=RANDOM_STATE)
X_train_pca6 = pca_6.fit_transform(X_train_full)
X_test_pca6  = pca_6.transform(X_test_full)

# PCA sonrası angle aralığına çek
pca_scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_pca6_angle = pca_scaler.fit_transform(X_train_pca6)
X_test_pca6_angle  = pca_scaler.transform(X_test_pca6)

aciklanan_varyans = pca_6.explained_variance_ratio_.sum() * 100
print(f"   PCA-6 train shape: {X_train_pca6_angle.shape}")
print(f"   Açıklanan varyans: %{aciklanan_varyans:.2f}")
print(f"   PCA-6 angle aralığı: [{X_train_pca6_angle.min():.4f}, {X_train_pca6_angle.max():.4f}]")

# ── Adım 6: Amplitude Encoding Hazırlığı (16-pad, 4-qubit) ──
print(f"\n📌 ADIM 6: Amplitude Encoding Hazırlığı")

# 10 özellik → 16 boyuta padding (2^4 = 16)
X_train_amp_padded = np.pad(X_train_full, ((0, 0), (0, 6)), mode='constant')
X_test_amp_padded  = np.pad(X_test_full,  ((0, 0), (0, 6)), mode='constant')

# L2 normalize (amplitude encoding için zorunlu)
train_norm = np.linalg.norm(X_train_amp_padded, axis=1, keepdims=True)
test_norm  = np.linalg.norm(X_test_amp_padded,  axis=1, keepdims=True)

# Sıfır norm sorunu varsa epsilon ekle
train_norm[train_norm == 0] = 1e-10
test_norm[test_norm == 0]   = 1e-10

X_train_amp = X_train_amp_padded / train_norm
X_test_amp  = X_test_amp_padded  / test_norm

print(f"   Amplitude shape   : {X_train_amp.shape} (16 boyut → 4 qubit)")
print(f"   L2 norm kontrol   : {np.linalg.norm(X_train_amp[0]):.6f} (≈1.0 olmalı)")

# ── Adım 7: Tüm Quantum Verileri Sözlüğe Topla ──────────────
QUANTUM_VERILER = {
    # Angle encoding (10 özellik, 10 qubit için)
    'X_train_angle10' : X_train_angle.astype(np.float32),
    'X_test_angle10'  : X_test_angle.astype(np.float32),

    # PCA-6 angle (6 qubit için, WBCD simetri)
    'X_train_pca6'    : X_train_pca6_angle.astype(np.float32),
    'X_test_pca6'     : X_test_pca6_angle.astype(np.float32),

    # Amplitude encoding (4 qubit için, 16-boyut padded)
    'X_train_amp16'   : X_train_amp.astype(np.float32),
    'X_test_amp16'    : X_test_amp.astype(np.float32),

    # Etiketler
    'y_train'         : y_train.astype(np.int64),
    'y_test'          : y_test.astype(np.int64),

    # Yardımcı bilgi
    'sayisal_scaler'  : sayisal_scaler,
    'pca_6'           : pca_6,
    'pca_scaler'      : pca_scaler,
    'pca_varyans'     : aciklanan_varyans,
}

print(f"\n💾 QUANTUM_VERILER sözlüğü oluşturuldu (11 anahtar)")
print(f"\n📌 QUANTUM VERSIYON ÖZETİ:")
print(f"   - Angle-10  : {QUANTUM_VERILER['X_train_angle10'].shape}  (10 qubit)")
print(f"   - PCA-6     : {QUANTUM_VERILER['X_train_pca6'].shape}  (6 qubit)")
print(f"   - Amp-16    : {QUANTUM_VERILER['X_train_amp16'].shape} (4 qubit)")

# ── Quantum Preprocessing Raporu Kaydet ─────────────────────
qprep_rapor = {
    "tarih"             : datetime.now().strftime('%d/%m/%Y %H:%M:%S'),
    "n_train"           : int(X_train_full.shape[0]),
    "n_test"            : int(X_test_full.shape[0]),
    "preprocessing"     : {
        "sayisal_scaler" : "StandardScaler (z-score)",
        "kategorik"      : "Manual ordinal mapping",
        "angle_mapping"  : "clip[-3,3] * π/3",
        "amplitude"      : "padding 10 → 16 + L2 normalize"
    },
    "quantum_versions": {
        "angle10"    : {"shape": list(X_train_angle.shape), "qubit": 10},
        "pca6"       : {"shape": list(X_train_pca6_angle.shape), "qubit": 6,
                        "varyans_yuzdesi": float(aciklanan_varyans)},
        "amp16"      : {"shape": list(X_train_amp.shape), "qubit": 4}
    }
}

rapor_yol = f'{OUTPUT_DIR}/sonuclar/preprocessing_quantum_rapor.json'
with open(rapor_yol, 'w', encoding='utf-8') as f:
    json.dump(qprep_rapor, f, ensure_ascii=False, indent=4)

print(f"\n💾 Rapor kaydedildi: {rapor_yol}")


In [ ]:
print("=" * 60)
print("🛠️  ORTAK FONKSİYONLAR")
print("=" * 60)

# ── Metrik Hesaplama (Multiclass) ───────────────────────────
def metrikleri_hesapla(y_true, y_pred, y_prob=None, model_adi=""):
    """
    Multiclass için tüm metrikleri hesaplar.

    Args:
        y_true: Gerçek etiketler (integer)
        y_pred: Tahmin edilen etiketler (integer)
        y_prob: Olasılık tahminleri (n_samples, n_classes) - opsiyonel
        model_adi: Model ismi

    Returns:
        Dict (tüm metrikler)
    """
    sonuc = {
        "model"            : model_adi,
        "accuracy"         : float(accuracy_score(y_true, y_pred)),
        "f1_macro"         : float(f1_score(y_true, y_pred, average='macro')),
        "f1_weighted"      : float(f1_score(y_true, y_pred, average='weighted')),
        "precision_macro"  : float(precision_score(y_true, y_pred,
                                                    average='macro', zero_division=0)),
        "recall_macro"     : float(recall_score(y_true, y_pred, average='macro')),
        "mcc"              : float(matthews_corrcoef(y_true, y_pred)),
        "kappa"            : float(cohen_kappa_score(y_true, y_pred)),
    }

    # AUC sadece olasılık varsa
    if y_prob is not None:
        try:
            sonuc["auc_macro"] = float(roc_auc_score(y_true, y_prob,
                                                     multi_class='ovr',
                                                     average='macro'))
            sonuc["auc_weighted"] = float(roc_auc_score(y_true, y_prob,
                                                        multi_class='ovr',
                                                        average='weighted'))
        except Exception as e:
            sonuc["auc_macro"] = None
            sonuc["auc_weighted"] = None

    return sonuc

# ── CV Sonuçlarını Özetle ───────────────────────────────────
def cv_ozetle(fold_sonuclari, model_adi):
    """
    5-fold CV sonuçlarının mean ± std özetini çıkarır.
    """
    metrik_isimleri = ['accuracy', 'f1_macro', 'f1_weighted',
                       'precision_macro', 'recall_macro', 'mcc', 'kappa']
    if 'auc_macro' in fold_sonuclari[0]:
        metrik_isimleri += ['auc_macro', 'auc_weighted']

    ozet = {"model": model_adi}
    for m in metrik_isimleri:
        degerler = [f[m] for f in fold_sonuclari if f.get(m) is not None]
        if len(degerler) > 0:
            ozet[f"{m}_mean"] = float(np.mean(degerler))
            ozet[f"{m}_std"]  = float(np.std(degerler))
        else:
            ozet[f"{m}_mean"] = None
            ozet[f"{m}_std"]  = None

    return ozet

# ── Sonuç Yazdırma ──────────────────────────────────────────
def sonuc_yazdir(ozet, sure=None):
    """
    Bir model özetini güzel bir formatta yazdırır.
    """
    print(f"\n{'='*55}")
    print(f"📊 SONUÇ ÖZETİ: {ozet['model']}")
    print(f"{'='*55}")

    print(f"   Accuracy        : {ozet['accuracy_mean']:.4f} ± {ozet['accuracy_std']:.4f}")
    print(f"   F1 (macro)      : {ozet['f1_macro_mean']:.4f} ± {ozet['f1_macro_std']:.4f}")
    print(f"   F1 (weighted)   : {ozet['f1_weighted_mean']:.4f} ± {ozet['f1_weighted_std']:.4f}")
    print(f"   Precision (mac) : {ozet['precision_macro_mean']:.4f} ± {ozet['precision_macro_std']:.4f}")
    print(f"   Recall (macro)  : {ozet['recall_macro_mean']:.4f} ± {ozet['recall_macro_std']:.4f}")
    print(f"   MCC             : {ozet['mcc_mean']:.4f} ± {ozet['mcc_std']:.4f}")
    print(f"   Kappa           : {ozet['kappa_mean']:.4f} ± {ozet['kappa_std']:.4f}")

    if 'auc_macro_mean' in ozet and ozet['auc_macro_mean'] is not None:
        print(f"   AUC (macro)     : {ozet['auc_macro_mean']:.4f} ± {ozet['auc_macro_std']:.4f}")

    if sure is not None:
        print(f"   Süre            : {sure:.1f} saniye ({sure/60:.1f} dk)")

# ── Confusion Matrix Çiz ────────────────────────────────────
def confusion_matrix_ciz(y_true, y_pred, model_adi, kaydet=True):
    """
    7-sınıflı multiclass confusion matrix.
    """
    cm = confusion_matrix(y_true, y_pred)
    label_encoder = KLASIK_VERILER['label_encoder']
    sinif_isimleri = label_encoder.classes_

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=sinif_isimleri,
                yticklabels=sinif_isimleri,
                cbar_kws={'label': 'Örnek Sayısı'})
    plt.xlabel('Tahmin Edilen Sınıf')
    plt.ylabel('Gerçek Sınıf')
    plt.title(f'Confusion Matrix — {model_adi}', fontweight='bold')
    plt.xticks(rotation=30, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()

    if kaydet:
        dosya = f'{GORSELLER}/cm_{model_adi}.png'
        plt.savefig(dosya, dpi=150, bbox_inches='tight')

    plt.show()

# ── Eğitim Eğrisi Çiz ───────────────────────────────────────
def egitim_egrisi_ciz(kayip_gecmisi, model_adi, kaydet=True):
    """
    Quantum modellerin epoch-bazlı kayıp eğrisi.
    """
    plt.figure(figsize=(8, 4))
    plt.plot(kayip_gecmisi, linewidth=2, color='#3498db')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Eğitim Kaybı — {model_adi}', fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    if kaydet:
        dosya = f'{GORSELLER}/loss_{model_adi}.png'
        plt.savefig(dosya, dpi=150, bbox_inches='tight')

    plt.show()

# ── Checkpoint Kaydet ───────────────────────────────────────
def checkpoint_kaydet(veri, dosya_adi, klasor):
    """
    Model sonuçlarını JSON ve PKL olarak kaydet.
    """
    # JSON (insan-okur)
    try:
        json_yol = f'{klasor}/{dosya_adi}.json'
        with open(json_yol, 'w', encoding='utf-8') as f:
            json.dump(veri, f, ensure_ascii=False, indent=4, default=str)
    except Exception as e:
        print(f"   ⚠️  JSON kaydetme uyarısı: {e}")

    # PKL (Python obje)
    pkl_yol = f'{klasor}/{dosya_adi}.pkl'
    with open(pkl_yol, 'wb') as f:
        pickle.dump(veri, f)

    print(f"   💾 Kaydedildi: {dosya_adi}.json + .pkl")

# ── Sonuç Listesine Ekle (TUM_SONUCLAR güncel tut) ──────────
TUM_SONUCLAR = {
    "klasik"  : [],
    "quantum" : [],
    "meta"    : {
        "baslangic"  : datetime.now().strftime('%d/%m/%Y %H:%M:%S'),
        "veri_seti"  : "UCI Obesity (Palechor 2019)",
        "n_ornek"    : 2087,
        "n_ozellik"  : 16,
        "n_sinif"    : 7,
        "cv_fold"    : 5,
        "random_state": RANDOM_STATE
    }
}

def sonuc_ekle(ozet, tur):
    """
    TUM_SONUCLAR sözlüğüne yeni model sonucu ekler.
    """
    TUM_SONUCLAR[tur].append(ozet)
    # Anlık snapshot kaydet
    snapshot_yol = f'{OUTPUT_DIR}/sonuclar/TUM_SONUCLAR_GUNCEL.json'
    with open(snapshot_yol, 'w', encoding='utf-8') as f:
        json.dump(TUM_SONUCLAR, f, ensure_ascii=False, indent=4, default=str)

print("\n✅ Tanımlanan fonksiyonlar:")
print("   1. metrikleri_hesapla()")
print("   2. cv_ozetle()")
print("   3. sonuc_yazdir()")
print("   4. confusion_matrix_ciz()")
print("   5. egitim_egrisi_ciz()")
print("   6. checkpoint_kaydet()")
print("   7. sonuc_ekle()")
print("\n📦 TUM_SONUCLAR sözlüğü hazır (klasik + quantum)")


In [ ]:
print("=" * 60)
print("🛠️  KLASİK MODELLER İÇİN ALTYAPI")
print("=" * 60)

from sklearn.base import clone

def klasik_egit(model_pipeline, param_grid, model_adi,
                X_train, y_train, X_test, y_test, n_features):
    """
    Klasik model için tam protokol:
      1. GridSearchCV ile hyperparameter tuning (5-fold inner)
      2. Best params ile 5-fold CV değerlendirmesi (outer)
      3. Test seti değerlendirmesi
      4. Tüm sonuçları kaydet
    """
    print(f"\n{'='*60}")
    print(f"🔵 KLASİK MODEL: {model_adi} ({n_features} özellik)")
    print(f"{'='*60}")

    baslangic = time.time()

    # ── Adım 1: GridSearchCV ile Tuning ─────────────────────
    print(f"\n📌 ADIM 1: GridSearchCV ile Hyperparameter Tuning...")
    n_combos = 1
    for v in param_grid.values():
        n_combos *= len(v)
    print(f"   Parametre kombinasyonları: {n_combos}")

    grid = GridSearchCV(
        estimator   = model_pipeline,
        param_grid  = param_grid,
        cv          = 5,
        scoring     = 'accuracy',
        n_jobs      = -1,
        verbose     = 0
    )
    grid.fit(X_train, y_train)

    print(f"   ✅ En iyi parametreler: {grid.best_params_}")
    print(f"   ✅ En iyi CV accuracy : {grid.best_score_:.4f}")

    best_model = grid.best_estimator_

    # ── Adım 2: 5-Fold Stratified CV Değerlendirmesi ────────
    print(f"\n📌 ADIM 2: 5-Fold Stratified CV Değerlendirmesi...")

    skf = KLASIK_VERILER['skf']
    fold_sonuclari = []

    for fold_no, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
        # Pandas DataFrame veya numpy array uyumluluğu
        if hasattr(X_train, 'iloc'):
            X_fold_tr  = X_train.iloc[tr_idx]
            X_fold_val = X_train.iloc[val_idx]
        else:
            X_fold_tr  = X_train[tr_idx]
            X_fold_val = X_train[val_idx]

        y_fold_tr  = y_train[tr_idx]
        y_fold_val = y_train[val_idx]

        # Best modeli klonla, fold'a fit et
        fold_model = clone(best_model)
        fold_model.fit(X_fold_tr, y_fold_tr)

        # Tahmin + olasılık
        y_pred_fold = fold_model.predict(X_fold_val)
        try:
            y_prob_fold = fold_model.predict_proba(X_fold_val)
        except:
            y_prob_fold = None

        fold_metrik = metrikleri_hesapla(
            y_fold_val, y_pred_fold, y_prob_fold,
            f"{model_adi}_fold{fold_no}"
        )
        fold_sonuclari.append(fold_metrik)

        print(f"   Fold {fold_no}: Acc={fold_metrik['accuracy']:.4f} "
              f"| F1m={fold_metrik['f1_macro']:.4f} "
              f"| MCC={fold_metrik['mcc']:.4f}")

    # ── Adım 3: CV Özeti ────────────────────────────────────
    ozet = cv_ozetle(fold_sonuclari, model_adi)
    ozet['n_features']    = n_features
    ozet['best_params']   = {k: str(v) for k, v in grid.best_params_.items()}
    ozet['cv_best_score'] = float(grid.best_score_)

    # ── Adım 4: Test Seti Değerlendirmesi ───────────────────
    print(f"\n📌 ADIM 3: Test Seti Değerlendirmesi...")
    best_model.fit(X_train, y_train)
    y_pred_test = best_model.predict(X_test)
    try:
        y_prob_test = best_model.predict_proba(X_test)
    except:
        y_prob_test = None

    test_metrik = metrikleri_hesapla(
        y_test, y_pred_test, y_prob_test, f"{model_adi}_test"
    )

    print(f"   Test Acc        : {test_metrik['accuracy']:.4f}")
    print(f"   Test F1 (macro) : {test_metrik['f1_macro']:.4f}")
    print(f"   Test F1 (wgt)   : {test_metrik['f1_weighted']:.4f}")
    print(f"   Test MCC        : {test_metrik['mcc']:.4f}")

    # ── Adım 5: Sonuç + Görsel + Kayıt ──────────────────────
    sure = time.time() - baslangic
    ozet['sure_sn'] = round(sure, 1)

    sonuc_yazdir(ozet, sure=sure)

    # Confusion matrix
    confusion_matrix_ciz(y_test, y_pred_test, f"klasik_{model_adi}_{n_features}f")

    # Kayıt
    kayit_paket = {
        "ozet"           : ozet,
        "fold_sonuclari" : fold_sonuclari,
        "test_metrik"    : test_metrik,
        "best_params"    : {k: str(v) for k, v in grid.best_params_.items()},
        "n_features"     : n_features
    }
    checkpoint_kaydet(kayit_paket, f"klasik_{model_adi}_{n_features}f", SONUC_KLASIK)
    sonuc_ekle(ozet, "klasik")

    return best_model, ozet, fold_sonuclari, test_metrik

# ── Model Pipeline ve Grid Tanımları ────────────────────────
print("\n📌 6 KLASİK MODEL HAZIRLANIYOR:")

# 1. LOGISTIC REGRESSION
def lr_pipeline_yap(preprocessor):
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(
            max_iter=2000, random_state=RANDOM_STATE
        ))
    ])

LR_GRID = {
    'model__C'      : [0.01, 0.1, 1.0, 10.0],
    'model__solver' : ['lbfgs', 'saga']
}

# 2. SVM-RBF
def svm_pipeline_yap(preprocessor):
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', SVC(
            kernel='rbf', probability=True, random_state=RANDOM_STATE
        ))
    ])

SVM_GRID = {
    'model__C'     : [0.1, 1.0, 10.0],
    'model__gamma' : ['scale', 0.01, 0.1]
}

# 3. RANDOM FOREST
def rf_pipeline_yap(preprocessor):
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])

RF_GRID = {
    'model__n_estimators'      : [100, 200, 500],
    'model__max_depth'         : [None, 10, 20],
    'model__min_samples_split' : [2, 5]
}

# 4. XGBOOST
def xgb_pipeline_yap(preprocessor):
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', xgb.XGBClassifier(
            objective='multi:softprob',
            num_class=N_CLASSES,
            eval_metric='mlogloss',
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

XGB_GRID = {
    'model__n_estimators'  : [100, 200],
    'model__max_depth'     : [3, 6, 9],
    'model__learning_rate' : [0.05, 0.1]
}

# 5. KNN
def knn_pipeline_yap(preprocessor):
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', KNeighborsClassifier(n_jobs=-1))
    ])

KNN_GRID = {
    'model__n_neighbors' : [3, 5, 7, 11, 15],
    'model__weights'     : ['uniform', 'distance'],
    'model__metric'      : ['euclidean', 'manhattan']
}

# 6. MLP
def mlp_pipeline_yap(preprocessor):
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', MLPClassifier(
            random_state=RANDOM_STATE, max_iter=500, early_stopping=True
        ))
    ])

MLP_GRID = {
    'model__hidden_layer_sizes' : [(64,), (128,), (64, 32)],
    'model__alpha'              : [0.0001, 0.001, 0.01],
    'model__learning_rate_init' : [0.001, 0.01]
}

print("   1. Logistic Regression (LR)  →  8 grid combo")
print("   2. SVM-RBF                   →  9 grid combo")
print("   3. Random Forest (RF)        → 18 grid combo")
print("   4. XGBoost                   → 12 grid combo")
print("   5. KNN                       → 20 grid combo")
print("   6. MLP                       → 18 grid combo")
print(f"\n   Toplam grid kombinasyonu  : 85")


In [ ]:
# ── 16-özellik versiyonu ────────────────────────────────────
print("\n--- LR — 16 ÖZELLİK ---")

lr_pipe_16 = lr_pipeline_yap(KLASIK_VERILER['preprocessor_16'])
lr_model_16, lr_ozet_16, lr_folds_16, lr_test_16 = klasik_egit(
    model_pipeline = lr_pipe_16,
    param_grid     = LR_GRID,
    model_adi      = "LogisticRegression",
    X_train        = KLASIK_VERILER['X_train_16'],
    y_train        = KLASIK_VERILER['y_train'],
    X_test         = KLASIK_VERILER['X_test_16'],
    y_test         = KLASIK_VERILER['y_test'],
    n_features     = 16
)

# ── 10-özellik versiyonu (Top-10) ───────────────────────────
print("\n--- LR — 10 ÖZELLİK (TOP-10) ---")

lr_pipe_10 = lr_pipeline_yap(KLASIK_VERILER['preprocessor_10'])
lr_model_10, lr_ozet_10, lr_folds_10, lr_test_10 = klasik_egit(
    model_pipeline = lr_pipe_10,
    param_grid     = LR_GRID,
    model_adi      = "LogisticRegression_top10",
    X_train        = KLASIK_VERILER['X_train_10'],
    y_train        = KLASIK_VERILER['y_train'],
    X_test         = KLASIK_VERILER['X_test_10'],
    y_test         = KLASIK_VERILER['y_test'],
    n_features     = 10
)


In [ ]:
# ── 16-özellik versiyonu ────────────────────────────────────
print("\n--- SVM-RBF — 16 ÖZELLİK ---")
print("SVM-RBF eğitimi başlıyor (en yavaş klasik model)")

svm_pipe_16 = svm_pipeline_yap(KLASIK_VERILER['preprocessor_16'])
svm_model_16, svm_ozet_16, svm_folds_16, svm_test_16 = klasik_egit(
    model_pipeline = svm_pipe_16,
    param_grid     = SVM_GRID,
    model_adi      = "SVM_RBF",
    X_train        = KLASIK_VERILER['X_train_16'],
    y_train        = KLASIK_VERILER['y_train'],
    X_test         = KLASIK_VERILER['X_test_16'],
    y_test         = KLASIK_VERILER['y_test'],
    n_features     = 16
)

# ── 10-özellik versiyonu (Top-10) ───────────────────────────
print("\n--- SVM-RBF — 10 ÖZELLİK (TOP-10) ---")

svm_pipe_10 = svm_pipeline_yap(KLASIK_VERILER['preprocessor_10'])
svm_model_10, svm_ozet_10, svm_folds_10, svm_test_10 = klasik_egit(
    model_pipeline = svm_pipe_10,
    param_grid     = SVM_GRID,
    model_adi      = "SVM_RBF_top10",
    X_train        = KLASIK_VERILER['X_train_10'],
    y_train        = KLASIK_VERILER['y_train'],
    X_test         = KLASIK_VERILER['X_test_10'],
    y_test         = KLASIK_VERILER['y_test'],
    n_features     = 10
)


In [ ]:
# ── 16-özellik versiyonu ────────────────────────────────────
print("\n--- RANDOM FOREST — 16 ÖZELLİK ---")

rf_pipe_16 = rf_pipeline_yap(KLASIK_VERILER['preprocessor_16'])
rf_model_16, rf_ozet_16, rf_folds_16, rf_test_16 = klasik_egit(
    model_pipeline = rf_pipe_16,
    param_grid     = RF_GRID,
    model_adi      = "RandomForest",
    X_train        = KLASIK_VERILER['X_train_16'],
    y_train        = KLASIK_VERILER['y_train'],
    X_test         = KLASIK_VERILER['X_test_16'],
    y_test         = KLASIK_VERILER['y_test'],
    n_features     = 16
)

# ── 10-özellik versiyonu (Top-10) ───────────────────────────
print("\n--- RANDOM FOREST — 10 ÖZELLİK (TOP-10) ---")

rf_pipe_10 = rf_pipeline_yap(KLASIK_VERILER['preprocessor_10'])
rf_model_10, rf_ozet_10, rf_folds_10, rf_test_10 = klasik_egit(
    model_pipeline = rf_pipe_10,
    param_grid     = RF_GRID,
    model_adi      = "RandomForest_top10",
    X_train        = KLASIK_VERILER['X_train_10'],
    y_train        = KLASIK_VERILER['y_train'],
    X_test         = KLASIK_VERILER['X_test_10'],
    y_test         = KLASIK_VERILER['y_test'],
    n_features     = 10
)


In [ ]:
# ── 16-özellik versiyonu ────────────────────────────────────
print("\n--- XGBOOST — 16 ÖZELLİK ---")

xgb_pipe_16 = xgb_pipeline_yap(KLASIK_VERILER['preprocessor_16'])
xgb_model_16, xgb_ozet_16, xgb_folds_16, xgb_test_16 = klasik_egit(
    model_pipeline = xgb_pipe_16,
    param_grid     = XGB_GRID,
    model_adi      = "XGBoost",
    X_train        = KLASIK_VERILER['X_train_16'],
    y_train        = KLASIK_VERILER['y_train'],
    X_test         = KLASIK_VERILER['X_test_16'],
    y_test         = KLASIK_VERILER['y_test'],
    n_features     = 16
)

# ── 10-özellik versiyonu (Top-10) ───────────────────────────
print("\n--- XGBOOST — 10 ÖZELLİK (TOP-10) ---")

xgb_pipe_10 = xgb_pipeline_yap(KLASIK_VERILER['preprocessor_10'])
xgb_model_10, xgb_ozet_10, xgb_folds_10, xgb_test_10 = klasik_egit(
    model_pipeline = xgb_pipe_10,
    param_grid     = XGB_GRID,
    model_adi      = "XGBoost_top10",
    X_train        = KLASIK_VERILER['X_train_10'],
    y_train        = KLASIK_VERILER['y_train'],
    X_test         = KLASIK_VERILER['X_test_10'],
    y_test         = KLASIK_VERILER['y_test'],
    n_features     = 10
)


In [ ]:
# ── 16-özellik versiyonu ────────────────────────────────────
print("\n--- KNN — 16 ÖZELLİK ---")

knn_pipe_16 = knn_pipeline_yap(KLASIK_VERILER['preprocessor_16'])
knn_model_16, knn_ozet_16, knn_folds_16, knn_test_16 = klasik_egit(
    model_pipeline = knn_pipe_16,
    param_grid     = KNN_GRID,
    model_adi      = "KNN",
    X_train        = KLASIK_VERILER['X_train_16'],
    y_train        = KLASIK_VERILER['y_train'],
    X_test         = KLASIK_VERILER['X_test_16'],
    y_test         = KLASIK_VERILER['y_test'],
    n_features     = 16
)

# ── 10-özellik versiyonu (Top-10) ───────────────────────────
print("\n--- KNN — 10 ÖZELLİK (TOP-10) ---")

knn_pipe_10 = knn_pipeline_yap(KLASIK_VERILER['preprocessor_10'])
knn_model_10, knn_ozet_10, knn_folds_10, knn_test_10 = klasik_egit(
    model_pipeline = knn_pipe_10,
    param_grid     = KNN_GRID,
    model_adi      = "KNN_top10",
    X_train        = KLASIK_VERILER['X_train_10'],
    y_train        = KLASIK_VERILER['y_train'],
    X_test         = KLASIK_VERILER['X_test_10'],
    y_test         = KLASIK_VERILER['y_test'],
    n_features     = 10
)


In [ ]:
# ── 16-özellik versiyonu ────────────────────────────────────
print("\n--- MLP — 16 ÖZELLİK ---")

mlp_pipe_16 = mlp_pipeline_yap(KLASIK_VERILER['preprocessor_16'])
mlp_model_16, mlp_ozet_16, mlp_folds_16, mlp_test_16 = klasik_egit(
    model_pipeline = mlp_pipe_16,
    param_grid     = MLP_GRID,
    model_adi      = "MLP",
    X_train        = KLASIK_VERILER['X_train_16'],
    y_train        = KLASIK_VERILER['y_train'],
    X_test         = KLASIK_VERILER['X_test_16'],
    y_test         = KLASIK_VERILER['y_test'],
    n_features     = 16
)

# ── 10-özellik versiyonu (Top-10) ───────────────────────────
print("\n--- MLP — 10 ÖZELLİK (TOP-10) ---")

mlp_pipe_10 = mlp_pipeline_yap(KLASIK_VERILER['preprocessor_10'])
mlp_model_10, mlp_ozet_10, mlp_folds_10, mlp_test_10 = klasik_egit(
    model_pipeline = mlp_pipe_10,
    param_grid     = MLP_GRID,
    model_adi      = "MLP_top10",
    X_train        = KLASIK_VERILER['X_train_10'],
    y_train        = KLASIK_VERILER['y_train'],
    X_test         = KLASIK_VERILER['X_test_10'],
    y_test         = KLASIK_VERILER['y_test'],
    n_features     = 10
)


In [ ]:
# 12 modelin (6 × 2 feature seti) toplu özeti

print("=" * 65)
print("📊 KLASİK MODELLER — KONSOLİDE EDİLMİŞ ÖZET TABLOSU")
print("=" * 65)

# ── Adım 1: Tüm Klasik Sonuçları Tabloya Çevir ──────────────
klasik_sonuclar = TUM_SONUCLAR['klasik']
print(f"\n📌 Toplam klasik model sonucu: {len(klasik_sonuclar)}")

# DataFrame oluştur
satirlar = []
for s in klasik_sonuclar:
    satirlar.append({
        'Model'        : s['model'],
        'N_Features'   : s.get('n_features', '-'),
        'CV_Acc_Mean'  : s['accuracy_mean'],
        'CV_Acc_Std'   : s['accuracy_std'],
        'CV_F1_macro'  : s['f1_macro_mean'],
        'CV_F1_wgt'    : s['f1_weighted_mean'],
        'CV_MCC'       : s['mcc_mean'],
        'CV_Kappa'     : s['kappa_mean'],
        'CV_AUC_macro' : s.get('auc_macro_mean', None),
        'Sure_sn'      : s.get('sure_sn', '-'),
        'Best_Params'  : str(s.get('best_params', '-'))
    })

klasik_df = pd.DataFrame(satirlar)

# CV accuracy'ye göre sırala (büyükten küçüğe)
klasik_df_sirali = klasik_df.sort_values(
    by='CV_Acc_Mean', ascending=False
).reset_index(drop=True)
klasik_df_sirali.index += 1  # 1'den başla

print(f"\n📊 SIRALI ÖZET (CV Accuracy):")
print("=" * 95)
display_df = klasik_df_sirali[[
    'Model', 'N_Features', 'CV_Acc_Mean', 'CV_Acc_Std',
    'CV_F1_macro', 'CV_MCC', 'CV_AUC_macro', 'Sure_sn'
]].copy()

# Sayısal sütunları formatla
for col in ['CV_Acc_Mean', 'CV_Acc_Std', 'CV_F1_macro',
            'CV_MCC', 'CV_AUC_macro']:
    display_df[col] = display_df[col].apply(
        lambda x: f"{x:.4f}" if pd.notnull(x) else "-"
    )

print(display_df.to_string())

# ── Adım 2: Şampiyon Belirleme ──────────────────────────────
sampiyon = klasik_df_sirali.iloc[0]
print(f"\n{'='*65}")
print(f"🏆 KLASİK ŞAMPİYON")
print(f"{'='*65}")
print(f"   Model       : {sampiyon['Model']}")
print(f"   N_Features  : {sampiyon['N_Features']}")
print(f"   CV Accuracy : {sampiyon['CV_Acc_Mean']:.4f} ± {sampiyon['CV_Acc_Std']:.4f}")
print(f"   CV F1 macro : {sampiyon['CV_F1_macro']:.4f}")
print(f"   CV AUC      : {sampiyon['CV_AUC_macro']:.4f}")
print(f"   Best Params : {sampiyon['Best_Params']}")

# ── Adım 3: 16-feature vs 10-feature Karşılaştırması ───────
print(f"\n{'='*65}")
print(f"📊 FEATURE SET ETKİSİ (16-özellik vs Top-10)")
print(f"{'='*65}")

# Model isimlerini eşleştir
model_isim_map = {
    'LogisticRegression'        : 'LogisticRegression_top10',
    'SVM_RBF'                   : 'SVM_RBF_top10',
    'RandomForest'              : 'RandomForest_top10',
    'XGBoost'                   : 'XGBoost_top10',
    'KNN'                       : 'KNN_top10',
    'MLP'                       : 'MLP_top10',
}

print(f"\n   {'Model':<22} {'16-feat':>10} {'10-feat':>10} {'Δ (10-16)':>12}")
print(f"   {'-'*22} {'-'*10} {'-'*10} {'-'*12}")

karsilastirma_satirlar = []
for m16, m10 in model_isim_map.items():
    row16 = klasik_df[klasik_df['Model'] == m16]
    row10 = klasik_df[klasik_df['Model'] == m10]

    if len(row16) > 0 and len(row10) > 0:
        acc16 = row16['CV_Acc_Mean'].values[0]
        acc10 = row10['CV_Acc_Mean'].values[0]
        delta = acc10 - acc16
        sembol = "▲" if delta > 0.001 else ("▼" if delta < -0.001 else "≈")
        print(f"   {m16:<22} {acc16:>10.4f} {acc10:>10.4f} {sembol} {delta:>+9.4f}")
        karsilastirma_satirlar.append({
            'Model'   : m16,
            'Acc_16'  : float(acc16),
            'Acc_10'  : float(acc10),
            'Delta'   : float(delta)
        })

# ── Adım 4: Görselleştirme - Bar Chart ─────────────────────
print(f"\n📊 GÖRSEL 1: Tüm Klasik Modeller (Bar Chart)")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Klasik Modeller — Karşılaştırmalı Performans (5-Fold CV)',
             fontsize=14, fontweight='bold')

# 16 feature
df_16 = klasik_df[klasik_df['N_Features'] == 16].sort_values('CV_Acc_Mean')
axes[0].barh(df_16['Model'], df_16['CV_Acc_Mean'],
             xerr=df_16['CV_Acc_Std'],
             color='#3498db', edgecolor='black', alpha=0.8,
             error_kw={'capsize': 4, 'capthick': 1.5})
axes[0].set_xlabel('CV Accuracy (Mean ± Std)')
axes[0].set_title('16 Özellik (Tam)', fontweight='bold')
axes[0].set_xlim([0.80, 1.00])
axes[0].grid(axis='x', alpha=0.3)
for i, (m, v, s) in enumerate(zip(df_16['Model'],
                                   df_16['CV_Acc_Mean'],
                                   df_16['CV_Acc_Std'])):
    axes[0].text(v + 0.003, i, f'{v:.4f}', va='center', fontsize=9)

# 10 feature
df_10 = klasik_df[klasik_df['N_Features'] == 10].sort_values('CV_Acc_Mean')
axes[1].barh(df_10['Model'], df_10['CV_Acc_Mean'],
             xerr=df_10['CV_Acc_Std'],
             color='#e74c3c', edgecolor='black', alpha=0.8,
             error_kw={'capsize': 4, 'capthick': 1.5})
axes[1].set_xlabel('CV Accuracy (Mean ± Std)')
axes[1].set_title('Top-10 Özellik (RF Importance)', fontweight='bold')
axes[1].set_xlim([0.80, 1.00])
axes[1].grid(axis='x', alpha=0.3)
for i, (m, v, s) in enumerate(zip(df_10['Model'],
                                   df_10['CV_Acc_Mean'],
                                   df_10['CV_Acc_Std'])):
    axes[1].text(v + 0.003, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
gorsel1_yol = f'{GORSELLER}/klasik_karsilastirma_bar.png'
plt.savefig(gorsel1_yol, dpi=150, bbox_inches='tight')
plt.show()
print(f"   💾 Kaydedildi: {gorsel1_yol}")

# ── Adım 5: Görselleştirme - Feature Set Delta ─────────────
print(f"\n📊 GÖRSEL 2: Feature Set Etkisi (Δ = 10-feat - 16-feat)")

fig, ax = plt.subplots(figsize=(10, 5))
karsilastirma_df = pd.DataFrame(karsilastirma_satirlar)
karsilastirma_df = karsilastirma_df.sort_values('Delta')

renkler = ['#27ae60' if d > 0 else '#c0392b' for d in karsilastirma_df['Delta']]
bars = ax.barh(karsilastirma_df['Model'], karsilastirma_df['Delta'] * 100,
               color=renkler, edgecolor='black', alpha=0.8)
ax.axvline(x=0, color='black', linewidth=1)
ax.set_xlabel('Δ Accuracy (%) — Top-10 minus 16-feature', fontweight='bold')
ax.set_title('Feature Selection Etkisi: Top-10 vs Tam-16',
             fontweight='bold')
ax.grid(axis='x', alpha=0.3)

for bar, d in zip(bars, karsilastirma_df['Delta']):
    width = bar.get_width()
    label_x = width + (0.05 if width >= 0 else -0.05)
    ha = 'left' if width >= 0 else 'right'
    ax.text(label_x, bar.get_y() + bar.get_height()/2,
            f'{d*100:+.2f}%', va='center', ha=ha,
            fontweight='bold', fontsize=10)

plt.tight_layout()
gorsel2_yol = f'{GORSELLER}/feature_set_etkisi.png'
plt.savefig(gorsel2_yol, dpi=150, bbox_inches='tight')
plt.show()
print(f"   💾 Kaydedildi: {gorsel2_yol}")

# ── Adım 6: CSV Kaydet (Makale Tablo I için) ────────────────
csv_yol = f'{OUTPUT_DIR}/sonuclar/KLASIK_OZET_TABLO.csv'
klasik_df_sirali.to_csv(csv_yol, index=False)
print(f"\n💾 Tablo CSV kaydedildi: {csv_yol}")

# ── Adım 7: JSON Konsolidasyon ──────────────────────────────
konsolidasyon = {
    "tarih"         : datetime.now().strftime('%d/%m/%Y %H:%M:%S'),
    "n_klasik_model": len(klasik_sonuclar),
    "sampiyon": {
        "model"       : sampiyon['Model'],
        "n_features"  : int(sampiyon['N_Features']),
        "cv_acc_mean" : float(sampiyon['CV_Acc_Mean']),
        "cv_acc_std"  : float(sampiyon['CV_Acc_Std']),
        "cv_f1_macro" : float(sampiyon['CV_F1_macro']),
        "cv_auc_macro": float(sampiyon['CV_AUC_macro']),
        "best_params" : sampiyon['Best_Params']
    },
    "feature_set_etkisi" : karsilastirma_satirlar,
    "tum_sonuclar"       : klasik_sonuclar
}

konsol_yol = f'{OUTPUT_DIR}/sonuclar/KLASIK_KONSOLIDASYON.json'
with open(konsol_yol, 'w', encoding='utf-8') as f:
    json.dump(konsolidasyon, f, ensure_ascii=False, indent=4, default=str)

print(f"💾 Konsolidasyon JSON: {konsol_yol}")

# ── Final Özet ──────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"{'='*65}")
print(f"   Toplam model       : {len(klasik_sonuclar)} (6 × 2 feature seti)")
print(f"   Şampiyon           : {sampiyon['Model']}")
print(f"   En iyi CV accuracy : {sampiyon['CV_Acc_Mean']:.4f}")

print(f"   - 7 quantum model planlandı")
print(f"   - WBCD'deki ReUpload-6q-3block + obezite Q3 hibrit + 4 baseline")


In [ ]:
print("Kuantum eğitim altyapısı yükleniyor...")

from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.utils.class_weight import compute_class_weight

# ── Class Weight Hesapla ────────────────────────────────────
def class_weights_hesapla(y_train):
    """Sınıf ağırlıklarını otomatik hesapla."""
    siniflar = np.unique(y_train)
    weights = compute_class_weight(
        class_weight='balanced',
        classes=siniflar,
        y=y_train
    )
    return torch.tensor(weights, dtype=torch.float32)

def seedleri_sabitle(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def quantum_fold_egit(model_fabrika, X_tr, y_tr, X_val, y_val,
                      n_epochs=100, lr=0.05, batch_size=32,
                      patience=15, fold_no=1, seed_offset=0,
                      X_tr_2=None, X_val_2=None,
                      class_weights=None, verbose_her=20):
    """
    Tek-fold quantum eğitimi — class weight + scheduler + early stopping.
    """
    seedleri_sabitle(RANDOM_STATE + seed_offset + fold_no)

    model = model_fabrika()
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=lr*0.1)

    # Tensor'a çevir
    X_tr_t  = torch.tensor(X_tr,  dtype=torch.float32)
    y_tr_t  = torch.tensor(y_tr,  dtype=torch.long)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.long)

    if X_tr_2 is not None:
        X_tr_2_t  = torch.tensor(X_tr_2,  dtype=torch.float32)
        X_val_2_t = torch.tensor(X_val_2, dtype=torch.float32)
        dataset = TensorDataset(X_tr_t, X_tr_2_t, y_tr_t)
    else:
        X_tr_2_t = X_val_2_t = None
        dataset = TensorDataset(X_tr_t, y_tr_t)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    bas = time.time()
    kayip_gecmisi = []
    val_acc_gecmisi = []

    # Early stopping tracking
    en_iyi_val_acc = 0.0
    en_iyi_epoch = 0
    en_iyi_state = None
    sabir_sayaci = 0

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        n_batch = 0

        for batch in loader:
            optimizer.zero_grad()
            if X_tr_2 is not None:
                xb, xb_2, yb = batch
                logits = model(xb, xb_2)
            else:
                xb, yb = batch
                logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batch += 1

        scheduler.step()
        avg_loss = epoch_loss / n_batch
        kayip_gecmisi.append(avg_loss)

        # Val accuracy hesapla (early stopping için)
        model.eval()
        with torch.no_grad():
            if X_tr_2 is not None:
                vl = model(X_val_t, X_val_2_t)
            else:
                vl = model(X_val_t)
            val_preds = torch.argmax(vl, dim=1).numpy()
            val_acc = accuracy_score(y_val, val_preds)
            val_acc_gecmisi.append(val_acc)

        # Early stopping kontrolü
        if val_acc > en_iyi_val_acc:
            en_iyi_val_acc = val_acc
            en_iyi_epoch = epoch + 1
            en_iyi_state = {k: v.clone() for k, v in model.state_dict().items()}
            sabir_sayaci = 0
        else:
            sabir_sayaci += 1

        if (epoch + 1) % verbose_her == 0:
            print(f"      E{epoch+1:3d}/{n_epochs}: loss={avg_loss:.4f} | "
                  f"val_acc={val_acc:.4f} | best={en_iyi_val_acc:.4f}",
                  end='', flush=True)

        # Early stop
        if sabir_sayaci >= patience:
            print(f"\n      ⏸ Early stop @ epoch {epoch+1} "
                  f"(best epoch={en_iyi_epoch})", end='', flush=True)
            break

    sure = time.time() - bas

    # En iyi state'i geri yükle
    if en_iyi_state is not None:
        model.load_state_dict(en_iyi_state)

    # Final validation metrik
    model.eval()
    with torch.no_grad():
        if X_tr_2 is not None:
            val_logits = model(X_val_t, X_val_2_t)
        else:
            val_logits = model(X_val_t)
        val_probs = torch.softmax(val_logits, dim=1).numpy()
        val_preds = torch.argmax(val_logits, dim=1).numpy()

    val_metrik = metrikleri_hesapla(
        y_val, val_preds, val_probs, f"fold{fold_no}"
    )

    return {'model': model, 'kayip_gecmisi': kayip_gecmisi,
            'val_acc_gecmisi': val_acc_gecmisi,
            'val_metrik': val_metrik, 'sure': sure,
            'en_iyi_epoch': en_iyi_epoch}

def quantum_5fold_cv(model_fabrika, model_adi, X_train_full, y_train_full,
                     X_test_full, y_test_full,
                     n_epochs=100, lr=0.05, batch_size=32, patience=15,
                     X_train_2_full=None, X_test_2_full=None,
                     use_class_weights=True, model_aciklama=""):
    """5-fold CV — yeni standart."""
    print(f"\n{'='*65}")
    print(f"⚛️  QUANTUM MODEL: {model_adi}")
    print(f"{'='*65}")
    if model_aciklama:
        print(f"📌 {model_aciklama}")

    skf = KLASIK_VERILER['skf']
    fold_sonuclari = []
    fold_kayiplari = []
    fold_sureleri  = []
    fold_best_epochs = []

    # Class weights
    if use_class_weights:
        cw = class_weights_hesapla(y_train_full)
        print(f"📌 Class weights: {cw.numpy().round(3).tolist()}")
    else:
        cw = None

    toplam_bas = time.time()

    for fold_no, (tr_idx, val_idx) in enumerate(
            skf.split(X_train_full, y_train_full), 1):

        print(f"\n  📁 Fold {fold_no}/5", flush=True)

        X_fold_tr  = X_train_full[tr_idx]
        y_fold_tr  = y_train_full[tr_idx]
        X_fold_val = X_train_full[val_idx]
        y_fold_val = y_train_full[val_idx]

        if X_train_2_full is not None:
            X_fold_tr_2  = X_train_2_full[tr_idx]
            X_fold_val_2 = X_train_2_full[val_idx]
        else:
            X_fold_tr_2 = X_fold_val_2 = None

        fold_sonuc = quantum_fold_egit(
            model_fabrika = model_fabrika,
            X_tr=X_fold_tr,    y_tr=y_fold_tr,
            X_val=X_fold_val,  y_val=y_fold_val,
            n_epochs=n_epochs, lr=lr,
            batch_size=batch_size, patience=patience,
            fold_no=fold_no,
            X_tr_2=X_fold_tr_2, X_val_2=X_fold_val_2,
            class_weights=cw, verbose_her=20
        )

        fm = fold_sonuc['val_metrik']
        print(f"\n     ✅ Acc={fm['accuracy']:.4f} | F1m={fm['f1_macro']:.4f} | "
              f"MCC={fm['mcc']:.4f} | best_ep={fold_sonuc['en_iyi_epoch']} "
              f"({fold_sonuc['sure']:.0f}s)")

        fold_sonuclari.append(fm)
        fold_kayiplari.append(fold_sonuc['kayip_gecmisi'])
        fold_sureleri.append(fold_sonuc['sure'])
        fold_best_epochs.append(fold_sonuc['en_iyi_epoch'])

        # Checkpoint
        cp_yol = f"{CHECKPOINT}/qcp_{model_adi}_fold{fold_no}.json"
        try:
            with open(cp_yol, 'w', encoding='utf-8') as f:
                json.dump({"fold": fold_no, "metrik": fm,
                          "sure": fold_sonuc['sure'],
                          "best_epoch": fold_sonuc['en_iyi_epoch']},
                         f, ensure_ascii=False, indent=2, default=str)
        except:
            pass

    toplam_sure = time.time() - toplam_bas

    ozet = cv_ozetle(fold_sonuclari, model_adi)
    ozet['n_epochs_max']   = n_epochs
    ozet['lr_initial']     = lr
    ozet['batch_size']     = batch_size
    ozet['patience']       = patience
    ozet['mean_best_epoch']= float(np.mean(fold_best_epochs))
    ozet['sure_sn']        = round(toplam_sure, 1)
    ozet['backend']        = 'default.qubit'
    ozet['interface']      = 'torch'
    ozet['diff_method']    = 'backprop'
    ozet['class_weighting']= use_class_weights
    ozet['scheduler']      = 'CosineAnnealingLR'

    sonuc_yazdir(ozet, sure=toplam_sure)

    # Test seti retrain (ortalama best_epoch ile)
    print(f"\n📌 Test Seti Değerlendirmesi:")
    avg_best_ep = int(np.mean(fold_best_epochs))
    print(f"   Ortalama best_epoch: {avg_best_ep} → bu kadar epoch ile retrain")

    seedleri_sabitle(RANDOM_STATE + 100)
    final_model = model_fabrika()
    if use_class_weights:
        final_criterion = nn.CrossEntropyLoss(weight=cw)
    else:
        final_criterion = nn.CrossEntropyLoss()
    final_optimizer = optim.Adam(final_model.parameters(), lr=lr)
    final_scheduler = CosineAnnealingLR(final_optimizer, T_max=avg_best_ep,
                                         eta_min=lr*0.1)

    X_tr_t = torch.tensor(X_train_full, dtype=torch.float32)
    y_tr_t = torch.tensor(y_train_full, dtype=torch.long)
    X_te_t = torch.tensor(X_test_full,  dtype=torch.float32)

    if X_train_2_full is not None:
        X_tr_2_t = torch.tensor(X_train_2_full, dtype=torch.float32)
        X_te_2_t = torch.tensor(X_test_2_full,  dtype=torch.float32)
        ds = TensorDataset(X_tr_t, X_tr_2_t, y_tr_t)
    else:
        X_tr_2_t = X_te_2_t = None
        ds = TensorDataset(X_tr_t, y_tr_t)

    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    for epoch in range(avg_best_ep):
        final_model.train()
        for batch in loader:
            final_optimizer.zero_grad()
            if X_train_2_full is not None:
                xb, xb2, yb = batch
                logits = final_model(xb, xb2)
            else:
                xb, yb = batch
                logits = final_model(xb)
            loss = final_criterion(logits, yb)
            loss.backward()
            final_optimizer.step()
        final_scheduler.step()

    final_model.eval()
    with torch.no_grad():
        if X_train_2_full is not None:
            test_logits = final_model(X_te_t, X_te_2_t)
        else:
            test_logits = final_model(X_te_t)
        test_probs = torch.softmax(test_logits, dim=1).numpy()
        test_preds = torch.argmax(test_logits, dim=1).numpy()

    test_metrik = metrikleri_hesapla(
        y_test_full, test_preds, test_probs, f"{model_adi}_test"
    )
    print(f"   Test Acc        : {test_metrik['accuracy']:.4f}")
    print(f"   Test F1 (macro) : {test_metrik['f1_macro']:.4f}")
    print(f"   Test MCC        : {test_metrik['mcc']:.4f}")

    confusion_matrix_ciz(y_test_full, test_preds, f"quantum_{model_adi}")
    egitim_egrisi_ciz(fold_kayiplari[0], model_adi)

    kayit_paket = {
        "ozet": ozet,
        "fold_sonuclari": fold_sonuclari,
        "test_metrik": test_metrik,
        "fold_kayiplari": fold_kayiplari,
        "fold_sureleri": fold_sureleri,
        "fold_best_epochs": fold_best_epochs,
        "model_bilgisi": {
            "n_epochs_max"   : n_epochs,
            "lr_initial"     : lr,
            "batch_size"     : batch_size,
            "patience"       : patience,
            "scheduler"      : "CosineAnnealingLR",
            "class_weighting": use_class_weights,
            "backend"        : "default.qubit",
            "interface"      : "torch",
            "diff_method"    : "backprop"
        }
    }
    checkpoint_kaydet(kayit_paket, f"quantum_{model_adi}", SONUC_QUANTUM)
    sonuc_ekle(ozet, "quantum")

    print(f"\n✅ {model_adi} TAMAMLANDI ({toplam_sure/60:.1f} dk)")

    return {
        'ozet': ozet,
        'fold_sonuclari': fold_sonuclari,
        'test_metrik': test_metrik,
        'fold_kayiplari': fold_kayiplari,
        'final_model': final_model
    }


In [ ]:
# 6 qubit, AngleEmbedding (RY), 3-layer variational, 64-hidden head
# Class weighting + Cosine scheduler + Early stopping

print("=" * 65)
print("⚛️  Q-Angle-6q-3L ")
print("=" * 65)

# ── Mimari Sabitleri ────────────────────────────────────────
N_QUBITS_A6   = 6
N_LAYERS_A6   = 3        # 2 → 3
N_HIDDEN_A6   = 64       # 32 → 64
DROPOUT_A6    = 0.3      # 0.2 → 0.3
N_EPOCHS_A6   = 100      # 60 → 100
LR_A6         = 0.05
BATCH_A6      = 32
PATIENCE_A6   = 15
MODEL_ADI_A6  = "Q_Angle_6q_3L"

# ── Quantum Device ──────────────────────────────────────────
dev_a6_v2 = qml.device('default.qubit', wires=N_QUBITS_A6)

# ── Quantum Devre ───────────────────────────────────────────
@qml.qnode(dev_a6_v2, interface='torch', diff_method='backprop')
def angle_6q_3l_circuit(inputs, weights):
    """
    Q-Angle-6q-3L devresi — 3-layer variational
    """
    # 1. Angle Encoding
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS_A6), rotation='Y')

    # 2. Variational Layers (3 layer)
    for layer in range(N_LAYERS_A6):
        for i in range(N_QUBITS_A6):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        for i in range(N_QUBITS_A6):
            qml.CNOT(wires=[i, (i + 1) % N_QUBITS_A6])

    # 3. Tüm qubit'lerden ölçüm
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS_A6)]

# ── Hibrit Sınıflandırıcı ───────────────────────────────────
class AngleHybridClassifierV2(nn.Module):
    """
    Q-Angle-6q-3L: 6 qubit, 3 layer, 64-hidden classical head
    """
    def __init__(self):
        super().__init__()
        weight_shapes = {"weights": (N_LAYERS_A6, N_QUBITS_A6, 2)}
        self.qlayer = qml.qnn.TorchLayer(angle_6q_3l_circuit, weight_shapes)
        self.classical_head = nn.Sequential(
            nn.Linear(N_QUBITS_A6, N_HIDDEN_A6),
            nn.ReLU(),
            nn.Dropout(DROPOUT_A6),
            nn.Linear(N_HIDDEN_A6, N_CLASSES)
        )

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
            single = True
        else:
            single = False
        q_out = self.qlayer(x)
        out = self.classical_head(q_out)
        if single:
            out = out.squeeze(0)
        return out

def angle_6q_3l_fabrika():
    return AngleHybridClassifierV2()

# ── Mimari Bilgisi ──────────────────────────────────────────
print(f"\n📌 MİMARİ:")
print(f"   Qubit Sayısı     : {N_QUBITS_A6}")
print(f"   Variational Layer: {N_LAYERS_A6} (eski: 2)")
print(f"   Encoding         : AngleEmbedding (RY)")
print(f"   Entanglement     : Ring CNOT")
print(f"   Classical Head   : Linear({N_QUBITS_A6}→{N_HIDDEN_A6}) + ReLU + "
      f"Dropout({DROPOUT_A6}) + Linear({N_HIDDEN_A6}→{N_CLASSES})")
print(f"   Epoch (max)      : {N_EPOCHS_A6}")
print(f"   Early Stop       : patience={PATIENCE_A6}")
print(f"   Learning Rate    : {LR_A6} (cosine annealing)")
print(f"   Batch Size       : {BATCH_A6}")

test_model = angle_6q_3l_fabrika()
n_params = sum(p.numel() for p in test_model.parameters())
n_q_params = N_LAYERS_A6 * N_QUBITS_A6 * 2
print(f"\n📌 PARAMETRE SAYISI:")
print(f"   Quantum  : {n_q_params}")
print(f"   Classical: {n_params - n_q_params}")
print(f"   Toplam   : {n_params}")

# ── Smoke Test ──────────────────────────────────────────────
print(f"\n📌 SMOKE TEST (batch=4):")
seedleri_sabitle(RANDOM_STATE)
test_input = torch.tensor(QUANTUM_VERILER['X_train_pca6'][:4],
                           dtype=torch.float32)
test_logits = test_model(test_input)
print(f"   Batch logits shape: {test_logits.shape}  (beklenen: torch.Size([4, 7]))")
if test_logits.shape == torch.Size([4, 7]):
    print(f"   ✅ Devre çalışıyor!")
else:
    raise ValueError(f"Shape hatalı: {test_logits.shape}")

# ── 5-Fold CV ──────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"🚀 5-FOLD CV BAŞLIYOR (~12-18 dk)")
print(f"{'='*65}")

q_a6_v2_sonuc = quantum_5fold_cv(
    model_fabrika      = angle_6q_3l_fabrika,
    model_adi          = MODEL_ADI_A6,
    X_train_full       = QUANTUM_VERILER['X_train_pca6'],
    y_train_full       = QUANTUM_VERILER['y_train'],
    X_test_full        = QUANTUM_VERILER['X_test_pca6'],
    y_test_full        = QUANTUM_VERILER['y_test'],
    n_epochs           = N_EPOCHS_A6,
    lr                 = LR_A6,
    batch_size         = BATCH_A6,
    patience           = PATIENCE_A6,
    use_class_weights  = True,
    model_aciklama     = "Pure angle encoding (PCA-6), 3-layer, class-weighted, scheduler"
)


In [ ]:
# 4 qubit, AmplitudeEmbedding (16-pad), 3-layer variational
# Class weighting + scheduler + early stopping

print("=" * 65)
print("⚛️  Q-Amplitude-4q-3L ")
print("=" * 65)

# ── Mimari Sabitleri ────────────────────────────────────────
N_QUBITS_AMP   = 4
N_LAYERS_AMP   = 3       # 2 → 3
N_HIDDEN_AMP   = 64      # 32 → 64
DROPOUT_AMP    = 0.3
N_EPOCHS_AMP   = 100
LR_AMP         = 0.05
BATCH_AMP      = 32
PATIENCE_AMP   = 15
MODEL_ADI_AMP  = "Q_Amplitude_4q_3L"

# ── Quantum Device ──────────────────────────────────────────
dev_amp_v2 = qml.device('default.qubit', wires=N_QUBITS_AMP)

# ── Quantum Devre ───────────────────────────────────────────
@qml.qnode(dev_amp_v2, interface='torch', diff_method='backprop')
def amplitude_4q_3l_circuit(inputs, weights):
    """
    Q-Amplitude-4q-3L: 16-dim padded vector → 4 qubit amplitude embedding,
    3-layer variational
    """
    # 1. Amplitude Encoding (zaten L2 normalize)
    qml.AmplitudeEmbedding(inputs, wires=range(N_QUBITS_AMP),
                            normalize=True)

    # 2. Variational Layers (3 layer)
    for layer in range(N_LAYERS_AMP):
        for i in range(N_QUBITS_AMP):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        for i in range(N_QUBITS_AMP):
            qml.CNOT(wires=[i, (i + 1) % N_QUBITS_AMP])

    # 3. Tüm qubit'lerden ölçüm
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS_AMP)]

# ── Hibrit Sınıflandırıcı ───────────────────────────────────
class AmplitudeHybridClassifierV2(nn.Module):
    def __init__(self):
        super().__init__()
        weight_shapes = {"weights": (N_LAYERS_AMP, N_QUBITS_AMP, 2)}
        self.qlayer = qml.qnn.TorchLayer(amplitude_4q_3l_circuit, weight_shapes)
        self.classical_head = nn.Sequential(
            nn.Linear(N_QUBITS_AMP, N_HIDDEN_AMP),
            nn.ReLU(),
            nn.Dropout(DROPOUT_AMP),
            nn.Linear(N_HIDDEN_AMP, N_CLASSES)
        )

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
            single = True
        else:
            single = False
        q_out = self.qlayer(x)
        out = self.classical_head(q_out)
        if single:
            out = out.squeeze(0)
        return out

def amplitude_4q_3l_fabrika():
    return AmplitudeHybridClassifierV2()

# ── Mimari Bilgisi ──────────────────────────────────────────
print(f"\n📌 MİMARİ:")
print(f"   Qubit Sayısı     : {N_QUBITS_AMP}")
print(f"   Variational Layer: {N_LAYERS_AMP} (eski: 2)")
print(f"   Encoding         : AmplitudeEmbedding (16-dim → 4 qubit)")
print(f"   Entanglement     : Ring CNOT")
print(f"   Classical Head   : Linear({N_QUBITS_AMP}→{N_HIDDEN_AMP}) + ReLU + "
      f"Dropout({DROPOUT_AMP}) + Linear({N_HIDDEN_AMP}→{N_CLASSES})")
print(f"   Epoch (max)      : {N_EPOCHS_AMP}")
print(f"   Early Stop       : patience={PATIENCE_AMP}")
print(f"   Learning Rate    : {LR_AMP} (cosine annealing)")
print(f"   Batch Size       : {BATCH_AMP}")

test_model = amplitude_4q_3l_fabrika()
n_params = sum(p.numel() for p in test_model.parameters())
n_q_params = N_LAYERS_AMP * N_QUBITS_AMP * 2
print(f"\n📌 PARAMETRE SAYISI:")
print(f"   Quantum  : {n_q_params}")
print(f"   Classical: {n_params - n_q_params}")
print(f"   Toplam   : {n_params}")

# ── Smoke Test ──────────────────────────────────────────────
print(f"\n📌 SMOKE TEST (batch=4):")
seedleri_sabitle(RANDOM_STATE)
test_input = torch.tensor(QUANTUM_VERILER['X_train_amp16'][:4],
                           dtype=torch.float32)
test_logits = test_model(test_input)
print(f"   Batch logits shape: {test_logits.shape}  (beklenen: torch.Size([4, 7]))")
if test_logits.shape == torch.Size([4, 7]):
    print(f"   ✅ Devre çalışıyor!")
else:
    raise ValueError(f"Shape hatalı: {test_logits.shape}")

# ── 5-Fold CV ──────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"🚀 5-FOLD CV BAŞLIYOR (~8-12 dk)")
print(f"{'='*65}")

q_amp_v2_sonuc = quantum_5fold_cv(
    model_fabrika      = amplitude_4q_3l_fabrika,
    model_adi          = MODEL_ADI_AMP,
    X_train_full       = QUANTUM_VERILER['X_train_amp16'],
    y_train_full       = QUANTUM_VERILER['y_train'],
    X_test_full        = QUANTUM_VERILER['X_test_amp16'],
    y_test_full        = QUANTUM_VERILER['y_test'],
    n_epochs           = N_EPOCHS_AMP,
    lr                 = LR_AMP,
    batch_size         = BATCH_AMP,
    patience           = PATIENCE_AMP,
    use_class_weights  = True,
    model_aciklama     = "Amplitude encoding (16→4 qubit), 3-layer, V2 standart"
)


In [ ]:
# 6 qubit, 1 re-uploading block (data + variational + entanglement)

print("=" * 65)
print("⚛️  Q-ReUpload-6q-1block (Ablation Alt)")
print("=" * 65)

# ── Mimari Sabitleri ────────────────────────────────────────
N_QUBITS_RU1   = 6
N_BLOCKS_RU1   = 1       # ablation alt sınır
N_HIDDEN_RU1   = 64
DROPOUT_RU1    = 0.3
N_EPOCHS_RU1   = 100
LR_RU1         = 0.05
BATCH_RU1      = 32
PATIENCE_RU1   = 15
MODEL_ADI_RU1  = "Q_ReUpload_6q_1block"

# ── Quantum Device ──────────────────────────────────────────
dev_ru1 = qml.device('default.qubit', wires=N_QUBITS_RU1)

# ── Quantum Devre — Re-uploading ────────────────────────────
@qml.qnode(dev_ru1, interface='torch', diff_method='backprop')
def reupload_6q_1block_circuit(inputs, weights):
    """
    Q-ReUpload-6q-1block:
      Her blokta:
        1) Data re-uploading (RY angle encoding her qubit'e)
        2) Trainable Rot (3 parametreli her qubit'e)
        3) Ring CNOT entanglement
    """
    for block in range(N_BLOCKS_RU1):
        # 1. Data re-uploading
        for i in range(N_QUBITS_RU1):
            qml.RY(inputs[..., i], wires=i)

        # 2. Trainable rotation (3 params per qubit)
        for i in range(N_QUBITS_RU1):
            qml.Rot(
                weights[block, i, 0],
                weights[block, i, 1],
                weights[block, i, 2],
                wires=i
            )

        # 3. Ring entanglement
        for i in range(N_QUBITS_RU1):
            qml.CNOT(wires=[i, (i + 1) % N_QUBITS_RU1])

    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS_RU1)]

# ── Hibrit Sınıflandırıcı ───────────────────────────────────
class ReUpload1BlockClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        weight_shapes = {"weights": (N_BLOCKS_RU1, N_QUBITS_RU1, 3)}
        self.qlayer = qml.qnn.TorchLayer(reupload_6q_1block_circuit, weight_shapes)
        self.classical_head = nn.Sequential(
            nn.Linear(N_QUBITS_RU1, N_HIDDEN_RU1),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RU1),
            nn.Linear(N_HIDDEN_RU1, N_CLASSES)
        )

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
            single = True
        else:
            single = False
        q_out = self.qlayer(x)
        out = self.classical_head(q_out)
        if single:
            out = out.squeeze(0)
        return out

def reupload_1block_fabrika():
    return ReUpload1BlockClassifier()

# ── Mimari Bilgisi ──────────────────────────────────────────
print(f"\n📌 MİMARİ:")
print(f"   Qubit Sayısı     : {N_QUBITS_RU1}")
print(f"   Re-uploading Block: {N_BLOCKS_RU1} (ablation alt)")
print(f"   Block içeriği    : Data Re-upload + Rot + Ring CNOT")
print(f"   Encoding         : Re-uploading (RY her blok)")
print(f"   Classical Head   : Linear({N_QUBITS_RU1}→{N_HIDDEN_RU1}) + ReLU + "
      f"Dropout({DROPOUT_RU1}) + Linear({N_HIDDEN_RU1}→{N_CLASSES})")
print(f"   Epoch (max)      : {N_EPOCHS_RU1}")
print(f"   Early Stop       : patience={PATIENCE_RU1}")
print(f"   Learning Rate    : {LR_RU1} (cosine annealing)")
print(f"   Batch Size       : {BATCH_RU1}")

test_model = reupload_1block_fabrika()
n_params = sum(p.numel() for p in test_model.parameters())
n_q_params = N_BLOCKS_RU1 * N_QUBITS_RU1 * 3
print(f"\n📌 PARAMETRE SAYISI:")
print(f"   Quantum  : {n_q_params}")
print(f"   Classical: {n_params - n_q_params}")
print(f"   Toplam   : {n_params}")

# ── Smoke Test ──────────────────────────────────────────────
print(f"\n📌 SMOKE TEST (batch=4):")
seedleri_sabitle(RANDOM_STATE)
test_input = torch.tensor(QUANTUM_VERILER['X_train_pca6'][:4],
                           dtype=torch.float32)
test_logits = test_model(test_input)
print(f"   Batch logits shape: {test_logits.shape}  (beklenen: torch.Size([4, 7]))")
if test_logits.shape == torch.Size([4, 7]):
    print(f"   ✅ Devre çalışıyor!")
else:
    raise ValueError(f"Shape hatalı: {test_logits.shape}")

# ── 5-Fold CV ──────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"🚀 5-FOLD CV BAŞLIYOR (~8-12 dk)")
print(f"{'='*65}")

q_ru1_sonuc = quantum_5fold_cv(
    model_fabrika      = reupload_1block_fabrika,
    model_adi          = MODEL_ADI_RU1,
    X_train_full       = QUANTUM_VERILER['X_train_pca6'],
    y_train_full       = QUANTUM_VERILER['y_train'],
    X_test_full        = QUANTUM_VERILER['X_test_pca6'],
    y_test_full        = QUANTUM_VERILER['y_test'],
    n_epochs           = N_EPOCHS_RU1,
    lr                 = LR_RU1,
    batch_size         = BATCH_RU1,
    patience           = PATIENCE_RU1,
    use_class_weights  = True,
    model_aciklama     = "Re-uploading 1 block (PCA-6), V2 standart, ablation alt"
)


In [ ]:
# 6 qubit, 2 re-uploading block (ablation orta)

print("=" * 65)
print("⚛️  Q-ReUpload-6q-2block (Ablation Orta)")
print("=" * 65)

# ── Mimari Sabitleri ────────────────────────────────────────
N_QUBITS_RU2   = 6
N_BLOCKS_RU2   = 2       # ablation orta
N_HIDDEN_RU2   = 64
DROPOUT_RU2    = 0.3
N_EPOCHS_RU2   = 100
LR_RU2         = 0.05
BATCH_RU2      = 32
PATIENCE_RU2   = 15
MODEL_ADI_RU2  = "Q_ReUpload_6q_2block"

# ── Quantum Device ──────────────────────────────────────────
dev_ru2 = qml.device('default.qubit', wires=N_QUBITS_RU2)

# ── Quantum Devre — Re-uploading 2 block ────────────────────
@qml.qnode(dev_ru2, interface='torch', diff_method='backprop')
def reupload_6q_2block_circuit(inputs, weights):
    """
    Q-ReUpload-6q-2block:
      Her blokta:
        1) Data re-uploading (RY her qubit'e)
        2) Trainable Rot (3 parametreli)
        3) Ring CNOT
    """
    for block in range(N_BLOCKS_RU2):
        # 1. Data re-uploading
        for i in range(N_QUBITS_RU2):
            qml.RY(inputs[..., i], wires=i)

        # 2. Trainable rotation
        for i in range(N_QUBITS_RU2):
            qml.Rot(
                weights[block, i, 0],
                weights[block, i, 1],
                weights[block, i, 2],
                wires=i
            )

        # 3. Ring entanglement
        for i in range(N_QUBITS_RU2):
            qml.CNOT(wires=[i, (i + 1) % N_QUBITS_RU2])

    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS_RU2)]

# ── Hibrit Sınıflandırıcı ───────────────────────────────────
class ReUpload2BlockClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        weight_shapes = {"weights": (N_BLOCKS_RU2, N_QUBITS_RU2, 3)}
        self.qlayer = qml.qnn.TorchLayer(reupload_6q_2block_circuit, weight_shapes)
        self.classical_head = nn.Sequential(
            nn.Linear(N_QUBITS_RU2, N_HIDDEN_RU2),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RU2),
            nn.Linear(N_HIDDEN_RU2, N_CLASSES)
        )

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
            single = True
        else:
            single = False
        q_out = self.qlayer(x)
        out = self.classical_head(q_out)
        if single:
            out = out.squeeze(0)
        return out

def reupload_2block_fabrika():
    return ReUpload2BlockClassifier()

# ── Mimari Bilgisi ──────────────────────────────────────────
print(f"\n📌 MİMARİ:")
print(f"   Qubit Sayısı     : {N_QUBITS_RU2}")
print(f"   Re-uploading Block: {N_BLOCKS_RU2} (ablation orta)")
print(f"   Block içeriği    : Data Re-upload + Rot + Ring CNOT")
print(f"   Encoding         : Re-uploading (RY her blok)")
print(f"   Classical Head   : Linear({N_QUBITS_RU2}→{N_HIDDEN_RU2}) + ReLU + "
      f"Dropout({DROPOUT_RU2}) + Linear({N_HIDDEN_RU2}→{N_CLASSES})")
print(f"   Epoch (max)      : {N_EPOCHS_RU2}")
print(f"   Early Stop       : patience={PATIENCE_RU2}")
print(f"   Learning Rate    : {LR_RU2} (cosine annealing)")
print(f"   Batch Size       : {BATCH_RU2}")

test_model = reupload_2block_fabrika()
n_params = sum(p.numel() for p in test_model.parameters())
n_q_params = N_BLOCKS_RU2 * N_QUBITS_RU2 * 3
print(f"\n📌 PARAMETRE SAYISI:")
print(f"   Quantum  : {n_q_params}")
print(f"   Classical: {n_params - n_q_params}")
print(f"   Toplam   : {n_params}")

# ── Smoke Test ──────────────────────────────────────────────
print(f"\n📌 SMOKE TEST (batch=4):")
seedleri_sabitle(RANDOM_STATE)
test_input = torch.tensor(QUANTUM_VERILER['X_train_pca6'][:4],
                           dtype=torch.float32)
test_logits = test_model(test_input)
print(f"   Batch logits shape: {test_logits.shape}  (beklenen: torch.Size([4, 7]))")
if test_logits.shape == torch.Size([4, 7]):
    print(f"   ✅ Devre çalışıyor!")
else:
    raise ValueError(f"Shape hatalı: {test_logits.shape}")

# ── 5-Fold CV ──────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"🚀 5-FOLD CV BAŞLIYOR (~10-15 dk)")
print(f"{'='*65}")

q_ru2_sonuc = quantum_5fold_cv(
    model_fabrika      = reupload_2block_fabrika,
    model_adi          = MODEL_ADI_RU2,
    X_train_full       = QUANTUM_VERILER['X_train_pca6'],
    y_train_full       = QUANTUM_VERILER['y_train'],
    X_test_full        = QUANTUM_VERILER['X_test_pca6'],
    y_test_full        = QUANTUM_VERILER['y_test'],
    n_epochs           = N_EPOCHS_RU2,
    lr                 = LR_RU2,
    batch_size         = BATCH_RU2,
    patience           = PATIENCE_RU2,
    use_class_weights  = True,
    model_aciklama     = "Re-uploading 2 block (PCA-6), V2 standart, ablation orta"
)


In [ ]:
# 6 qubit, 3 re-uploading block (ablation üst, champion adayı)

print("=" * 65)
print("⚛️  Q-ReUpload-6q-3block (Ablation Üst)")
print("=" * 65)

# ── Mimari Sabitleri ────────────────────────────────────────
N_QUBITS_RU3   = 6
N_BLOCKS_RU3   = 3       # ablation üst, champion adayı
N_HIDDEN_RU3   = 64
DROPOUT_RU3    = 0.3
N_EPOCHS_RU3   = 100
LR_RU3         = 0.05
BATCH_RU3      = 32
PATIENCE_RU3   = 15
MODEL_ADI_RU3  = "Q_ReUpload_6q_3block"

# ── Quantum Device ──────────────────────────────────────────
dev_ru3 = qml.device('default.qubit', wires=N_QUBITS_RU3)

# ── Quantum Devre — Re-uploading 3 block ────────────────────
@qml.qnode(dev_ru3, interface='torch', diff_method='backprop')
def reupload_6q_3block_circuit(inputs, weights):
    """
    Q-ReUpload-6q-3block (Champion Aday):
      Her blokta:
        1) Data re-uploading (RY her qubit'e)
        2) Trainable Rot (3 parametreli)
        3) Ring CNOT
    Toplam 3 blok = en derin re-uploading varyantı.
    """
    for block in range(N_BLOCKS_RU3):
        # 1. Data re-uploading
        for i in range(N_QUBITS_RU3):
            qml.RY(inputs[..., i], wires=i)

        # 2. Trainable rotation
        for i in range(N_QUBITS_RU3):
            qml.Rot(
                weights[block, i, 0],
                weights[block, i, 1],
                weights[block, i, 2],
                wires=i
            )

        # 3. Ring entanglement
        for i in range(N_QUBITS_RU3):
            qml.CNOT(wires=[i, (i + 1) % N_QUBITS_RU3])

    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS_RU3)]

# ── Hibrit Sınıflandırıcı ───────────────────────────────────
class ReUpload3BlockClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        weight_shapes = {"weights": (N_BLOCKS_RU3, N_QUBITS_RU3, 3)}
        self.qlayer = qml.qnn.TorchLayer(reupload_6q_3block_circuit, weight_shapes)
        self.classical_head = nn.Sequential(
            nn.Linear(N_QUBITS_RU3, N_HIDDEN_RU3),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RU3),
            nn.Linear(N_HIDDEN_RU3, N_CLASSES)
        )

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
            single = True
        else:
            single = False
        q_out = self.qlayer(x)
        out = self.classical_head(q_out)
        if single:
            out = out.squeeze(0)
        return out

def reupload_3block_fabrika():
    return ReUpload3BlockClassifier()

# ── Mimari Bilgisi ──────────────────────────────────────────
print(f"\n📌 MİMARİ:")
print(f"   Qubit Sayısı     : {N_QUBITS_RU3}")
print(f"   Re-uploading Block: {N_BLOCKS_RU3} (ablation üst — champion aday)")
print(f"   Block içeriği    : Data Re-upload + Rot + Ring CNOT")
print(f"   Encoding         : Re-uploading (RY her blok)")
print(f"   Classical Head   : Linear({N_QUBITS_RU3}→{N_HIDDEN_RU3}) + ReLU + "
      f"Dropout({DROPOUT_RU3}) + Linear({N_HIDDEN_RU3}→{N_CLASSES})")
print(f"   Epoch (max)      : {N_EPOCHS_RU3}")
print(f"   Early Stop       : patience={PATIENCE_RU3}")
print(f"   Learning Rate    : {LR_RU3} (cosine annealing)")
print(f"   Batch Size       : {BATCH_RU3}")

test_model = reupload_3block_fabrika()
n_params = sum(p.numel() for p in test_model.parameters())
n_q_params = N_BLOCKS_RU3 * N_QUBITS_RU3 * 3
print(f"\n📌 PARAMETRE SAYISI:")
print(f"   Quantum  : {n_q_params}")
print(f"   Classical: {n_params - n_q_params}")
print(f"   Toplam   : {n_params}")

# ── Smoke Test ──────────────────────────────────────────────
print(f"\n📌 SMOKE TEST (batch=4):")
seedleri_sabitle(RANDOM_STATE)
test_input = torch.tensor(QUANTUM_VERILER['X_train_pca6'][:4],
                           dtype=torch.float32)
test_logits = test_model(test_input)
print(f"   Batch logits shape: {test_logits.shape}  (beklenen: torch.Size([4, 7]))")
if test_logits.shape == torch.Size([4, 7]):
    print(f"   ✅ Devre çalışıyor!")
else:
    raise ValueError(f"Shape hatalı: {test_logits.shape}")

# ── 5-Fold CV ──────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"🚀 5-FOLD CV BAŞLIYOR (~14-20 dk)")
print(f"{'='*65}")

q_ru3_sonuc = quantum_5fold_cv(
    model_fabrika      = reupload_3block_fabrika,
    model_adi          = MODEL_ADI_RU3,
    X_train_full       = QUANTUM_VERILER['X_train_pca6'],
    y_train_full       = QUANTUM_VERILER['y_train'],
    X_test_full        = QUANTUM_VERILER['X_test_pca6'],
    y_test_full        = QUANTUM_VERILER['y_test'],
    n_epochs           = N_EPOCHS_RU3,
    lr                 = LR_RU3,
    batch_size         = BATCH_RU3,
    patience           = PATIENCE_RU3,
    use_class_weights  = True,
    model_aciklama     = "Re-uploading 3 block (PCA-6), V2 standart, ablation üst — champion aday"
)


In [ ]:
# Branch 1: Angle (PCA-6, 6 qubit, 2 layer)
# Branch 2: Amplitude (16-pad, 4 qubit, 2 layer)
# Concat: [6+4] = 10 → classical head

print("=" * 65)
print("⚛️  Q-Hybrid-Q3 (Dual-Branch)")
print("=" * 65)

# ── Mimari Sabitleri ────────────────────────────────────────
# Branch 1: Angle
ANGLE_QUBITS_Q3   = 6
ANGLE_LAYERS_Q3   = 2

# Branch 2: Amplitude
AMP_QUBITS_Q3     = 4
AMP_LAYERS_Q3     = 2

# Classical head
N_HIDDEN_Q3       = 64
DROPOUT_Q3        = 0.3
N_EPOCHS_Q3       = 100
LR_Q3             = 0.05
BATCH_Q3          = 32
PATIENCE_Q3       = 15
MODEL_ADI_Q3      = "Q_Hybrid_Q3_DualBranch"

# ── Quantum Devices (iki ayrı device) ──────────────────────
dev_q3_angle = qml.device('default.qubit', wires=ANGLE_QUBITS_Q3)
dev_q3_amp   = qml.device('default.qubit', wires=AMP_QUBITS_Q3)

# ── Branch 1: Angle Quantum Devre ───────────────────────────
@qml.qnode(dev_q3_angle, interface='torch', diff_method='backprop')
def q3_angle_branch(inputs, weights):
    """
    Q3 Angle Branch: 6-qubit angle encoding + 2-layer variational
    """
    qml.AngleEmbedding(inputs, wires=range(ANGLE_QUBITS_Q3), rotation='Y')

    for layer in range(ANGLE_LAYERS_Q3):
        for i in range(ANGLE_QUBITS_Q3):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        for i in range(ANGLE_QUBITS_Q3):
            qml.CNOT(wires=[i, (i + 1) % ANGLE_QUBITS_Q3])

    return [qml.expval(qml.PauliZ(i)) for i in range(ANGLE_QUBITS_Q3)]

# ── Branch 2: Amplitude Quantum Devre ───────────────────────
@qml.qnode(dev_q3_amp, interface='torch', diff_method='backprop')
def q3_amp_branch(inputs, weights):
    """
    Q3 Amplitude Branch: 4-qubit amplitude encoding + 2-layer variational
    """
    qml.AmplitudeEmbedding(inputs, wires=range(AMP_QUBITS_Q3), normalize=True)

    for layer in range(AMP_LAYERS_Q3):
        for i in range(AMP_QUBITS_Q3):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        for i in range(AMP_QUBITS_Q3):
            qml.CNOT(wires=[i, (i + 1) % AMP_QUBITS_Q3])

    return [qml.expval(qml.PauliZ(i)) for i in range(AMP_QUBITS_Q3)]

# ── Hibrit Sınıflandırıcı (DUAL-BRANCH) ─────────────────────
class HybridQ3DualBranchClassifier(nn.Module):
    """
    Q-Hybrid-Q3: Ekibin özgün dual-branch quantum mimarisi.

    İki paralel quantum branch:
      - Angle branch (6 qubit) — semantik PCA-6 verisi
      - Amplitude branch (4 qubit) — bilgi-yoğun 16-dim padded vector

    İki branch'in PauliZ ölçümleri concat'lenir [6+4=10 boyut],
    classical head ile 7 sınıfa map'lenir.
    """
    def __init__(self):
        super().__init__()

        # Branch 1
        angle_weight_shapes = {"weights": (ANGLE_LAYERS_Q3, ANGLE_QUBITS_Q3, 2)}
        self.angle_qlayer = qml.qnn.TorchLayer(q3_angle_branch, angle_weight_shapes)

        # Branch 2
        amp_weight_shapes = {"weights": (AMP_LAYERS_Q3, AMP_QUBITS_Q3, 2)}
        self.amp_qlayer = qml.qnn.TorchLayer(q3_amp_branch, amp_weight_shapes)

        # Classical head: [6+4=10] → 64 → 7
        n_concat = ANGLE_QUBITS_Q3 + AMP_QUBITS_Q3
        self.classical_head = nn.Sequential(
            nn.Linear(n_concat, N_HIDDEN_Q3),
            nn.ReLU(),
            nn.Dropout(DROPOUT_Q3),
            nn.Linear(N_HIDDEN_Q3, N_CLASSES)
        )

    def forward(self, x_angle, x_amp):
        # Single-sample / batch handling
        angle_single = (x_angle.dim() == 1)
        amp_single = (x_amp.dim() == 1)

        if angle_single:
            x_angle = x_angle.unsqueeze(0)
        if amp_single:
            x_amp = x_amp.unsqueeze(0)

        angle_out = self.angle_qlayer(x_angle)  # (batch, 6)
        amp_out   = self.amp_qlayer(x_amp)      # (batch, 4)
        fused     = torch.cat([angle_out, amp_out], dim=1)  # (batch, 10)
        logits    = self.classical_head(fused)  # (batch, 7)

        if angle_single and amp_single:
            logits = logits.squeeze(0)
        return logits

def hybrid_q3_fabrika():
    return HybridQ3DualBranchClassifier()

# ── Mimari Bilgisi ──────────────────────────────────────────
print(f"\n📌 MİMARİ (DUAL-BRANCH):")
print(f"   Branch 1 (Angle):")
print(f"     Qubit       : {ANGLE_QUBITS_Q3}")
print(f"     Layer       : {ANGLE_LAYERS_Q3}")
print(f"     Encoding    : AngleEmbedding (RY) — PCA-6 verisi")
print(f"   Branch 2 (Amplitude):")
print(f"     Qubit       : {AMP_QUBITS_Q3}")
print(f"     Layer       : {AMP_LAYERS_Q3}")
print(f"     Encoding    : AmplitudeEmbedding — 16-dim padded")
print(f"   Concat       : [{ANGLE_QUBITS_Q3}+{AMP_QUBITS_Q3}] = "
      f"{ANGLE_QUBITS_Q3+AMP_QUBITS_Q3} boyut")
print(f"   Classical Head: Linear(10→{N_HIDDEN_Q3}) + ReLU + "
      f"Dropout({DROPOUT_Q3}) + Linear({N_HIDDEN_Q3}→{N_CLASSES})")
print(f"   Epoch (max)   : {N_EPOCHS_Q3}")
print(f"   Early Stop    : patience={PATIENCE_Q3}")
print(f"   Learning Rate : {LR_Q3} (cosine annealing)")
print(f"   Batch Size    : {BATCH_Q3}")

test_model = hybrid_q3_fabrika()
n_params = sum(p.numel() for p in test_model.parameters())
n_q_angle = ANGLE_LAYERS_Q3 * ANGLE_QUBITS_Q3 * 2
n_q_amp   = AMP_LAYERS_Q3 * AMP_QUBITS_Q3 * 2
n_q_total = n_q_angle + n_q_amp
print(f"\n📌 PARAMETRE SAYISI:")
print(f"   Quantum (angle branch): {n_q_angle}")
print(f"   Quantum (amp branch)  : {n_q_amp}")
print(f"   Quantum toplam        : {n_q_total}")
print(f"   Classical             : {n_params - n_q_total}")
print(f"   GENEL TOPLAM          : {n_params}")

# ── Smoke Test (DUAL INPUT) ────────────────────────────────
print(f"\n📌 SMOKE TEST (batch=4, dual input):")
seedleri_sabitle(RANDOM_STATE)
test_x_angle = torch.tensor(QUANTUM_VERILER['X_train_pca6'][:4],
                             dtype=torch.float32)
test_x_amp   = torch.tensor(QUANTUM_VERILER['X_train_amp16'][:4],
                             dtype=torch.float32)
test_logits = test_model(test_x_angle, test_x_amp)
print(f"   Angle input shape: {test_x_angle.shape}")
print(f"   Amp input shape  : {test_x_amp.shape}")
print(f"   Output logits    : {test_logits.shape}  (beklenen: torch.Size([4, 7]))")
if test_logits.shape == torch.Size([4, 7]):
    print(f"   ✅ Dual-branch çalışıyor!")
else:
    raise ValueError(f"Shape hatalı: {test_logits.shape}")

# ── 5-Fold CV (DUAL INPUT) ─────────────────────────────────
print(f"\n{'='*65}")
print(f"🚀 5-FOLD CV BAŞLIYOR (~18-25 dk)")
print(f"   ⚠️ Bu en uzun model, dual-branch nedeniyle 2x süre")
print(f"{'='*65}")

q_q3_sonuc = quantum_5fold_cv(
    model_fabrika      = hybrid_q3_fabrika,
    model_adi          = MODEL_ADI_Q3,
    X_train_full       = QUANTUM_VERILER['X_train_pca6'],
    y_train_full       = QUANTUM_VERILER['y_train'],
    X_test_full        = QUANTUM_VERILER['X_test_pca6'],
    y_test_full        = QUANTUM_VERILER['y_test'],
    X_train_2_full     = QUANTUM_VERILER['X_train_amp16'],
    X_test_2_full      = QUANTUM_VERILER['X_test_amp16'],
    n_epochs           = N_EPOCHS_Q3,
    lr                 = LR_Q3,
    batch_size         = BATCH_Q3,
    patience           = PATIENCE_Q3,
    use_class_weights  = True,
    model_aciklama     = "Dual-branch hybrid: Angle(6q,2L) + Amplitude(4q,2L) + concat[10] head"
)


In [ ]:
# Branch 1: ReUpload-6q-2block (Q3'teki Angle yerine, daha güçlü)
# Branch 2: Amplitude-4q-2L (aynen)
# Concat: [6+4] = 10 → classical head

print("=" * 65)
print("⚛️  Q-Hybrid-Q3-Plus (Re-upload Enhanced)")
print("=" * 65)

# ── Mimari Sabitleri ────────────────────────────────────────
# Branch 1: Re-upload (yeni)
RU_QUBITS_Q3P    = 6
RU_BLOCKS_Q3P    = 2

# Branch 2: Amplitude (aynı)
AMP_QUBITS_Q3P   = 4
AMP_LAYERS_Q3P   = 2

# Classical head
N_HIDDEN_Q3P     = 64
DROPOUT_Q3P      = 0.3
N_EPOCHS_Q3P     = 100
LR_Q3P           = 0.05
BATCH_Q3P        = 32
PATIENCE_Q3P     = 15
MODEL_ADI_Q3P    = "Q_Hybrid_Q3Plus_ReUpAmp"

# ── Quantum Devices ────────────────────────────────────────
dev_q3p_ru  = qml.device('default.qubit', wires=RU_QUBITS_Q3P)
dev_q3p_amp = qml.device('default.qubit', wires=AMP_QUBITS_Q3P)

# ── Branch 1: Re-upload Quantum Devre ──────────────────────
@qml.qnode(dev_q3p_ru, interface='torch', diff_method='backprop')
def q3plus_reupload_branch(inputs, weights):
    """Q3+ Branch 1: 6-qubit, 2-block re-uploading"""
    for block in range(RU_BLOCKS_Q3P):
        for i in range(RU_QUBITS_Q3P):
            qml.RY(inputs[..., i], wires=i)
        for i in range(RU_QUBITS_Q3P):
            qml.Rot(weights[block, i, 0], weights[block, i, 1],
                    weights[block, i, 2], wires=i)
        for i in range(RU_QUBITS_Q3P):
            qml.CNOT(wires=[i, (i + 1) % RU_QUBITS_Q3P])

    return [qml.expval(qml.PauliZ(i)) for i in range(RU_QUBITS_Q3P)]

# ── Branch 2: Amplitude (Q3 ile aynı) ──────────────────────
@qml.qnode(dev_q3p_amp, interface='torch', diff_method='backprop')
def q3plus_amp_branch(inputs, weights):
    """Q3+ Branch 2: 4-qubit amplitude, 2-layer (Q3 ile aynı)"""
    qml.AmplitudeEmbedding(inputs, wires=range(AMP_QUBITS_Q3P), normalize=True)

    for layer in range(AMP_LAYERS_Q3P):
        for i in range(AMP_QUBITS_Q3P):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        for i in range(AMP_QUBITS_Q3P):
            qml.CNOT(wires=[i, (i + 1) % AMP_QUBITS_Q3P])

    return [qml.expval(qml.PauliZ(i)) for i in range(AMP_QUBITS_Q3P)]

# ── Hibrit Sınıflandırıcı ───────────────────────────────────
class HybridQ3PlusClassifier(nn.Module):
    """
    Q-Hybrid-Q3-Plus: Q3'ün re-upload enhanced versiyonu.

    Branch 1: ReUpload-6q-2block (Q3'te Angle-2L vardı)
    Branch 2: Amplitude-4q-2L (Q3 ile aynı)
    Concat → Linear(10→64) → ReLU → Dropout → Linear(64→7)
    """
    def __init__(self):
        super().__init__()

        ru_weight_shapes = {"weights": (RU_BLOCKS_Q3P, RU_QUBITS_Q3P, 3)}
        self.ru_qlayer = qml.qnn.TorchLayer(q3plus_reupload_branch, ru_weight_shapes)

        amp_weight_shapes = {"weights": (AMP_LAYERS_Q3P, AMP_QUBITS_Q3P, 2)}
        self.amp_qlayer = qml.qnn.TorchLayer(q3plus_amp_branch, amp_weight_shapes)

        n_concat = RU_QUBITS_Q3P + AMP_QUBITS_Q3P  # 6+4=10
        self.classical_head = nn.Sequential(
            nn.Linear(n_concat, N_HIDDEN_Q3P),
            nn.ReLU(),
            nn.Dropout(DROPOUT_Q3P),
            nn.Linear(N_HIDDEN_Q3P, N_CLASSES)
        )

    def forward(self, x_pca, x_amp):
        pca_single = (x_pca.dim() == 1)
        amp_single = (x_amp.dim() == 1)

        if pca_single:
            x_pca = x_pca.unsqueeze(0)
        if amp_single:
            x_amp = x_amp.unsqueeze(0)

        ru_out  = self.ru_qlayer(x_pca)   # (batch, 6)
        amp_out = self.amp_qlayer(x_amp)  # (batch, 4)
        fused   = torch.cat([ru_out, amp_out], dim=1)  # (batch, 10)
        logits  = self.classical_head(fused)

        if pca_single and amp_single:
            logits = logits.squeeze(0)
        return logits

def hybrid_q3plus_fabrika():
    return HybridQ3PlusClassifier()

# ── Mimari Bilgisi ──────────────────────────────────────────
print(f"\n📌 MİMARİ (DUAL-BRANCH, RE-UPLOAD ENHANCED):")
print(f"   Branch 1 (Re-Upload — YENİ):")
print(f"     Qubit       : {RU_QUBITS_Q3P}")
print(f"     Block       : {RU_BLOCKS_Q3P}")
print(f"     Encoding    : Re-uploading (RY her blok)")
print(f"   Branch 2 (Amplitude — Q3 ile aynı):")
print(f"     Qubit       : {AMP_QUBITS_Q3P}")
print(f"     Layer       : {AMP_LAYERS_Q3P}")
print(f"     Encoding    : AmplitudeEmbedding")
print(f"   Concat       : [{RU_QUBITS_Q3P}+{AMP_QUBITS_Q3P}] = 10")
print(f"   Classical    : Linear(10→{N_HIDDEN_Q3P}) + ReLU + "
      f"Dropout({DROPOUT_Q3P}) + Linear({N_HIDDEN_Q3P}→{N_CLASSES})")
print(f"   Epoch (max)  : {N_EPOCHS_Q3P}")
print(f"   Early Stop   : patience={PATIENCE_Q3P}")
print(f"   Learning Rate: {LR_Q3P} (cosine annealing)")
print(f"   Batch Size   : {BATCH_Q3P}")

test_model = hybrid_q3plus_fabrika()
n_params = sum(p.numel() for p in test_model.parameters())
n_q_ru  = RU_BLOCKS_Q3P * RU_QUBITS_Q3P * 3
n_q_amp = AMP_LAYERS_Q3P * AMP_QUBITS_Q3P * 2
n_q_total = n_q_ru + n_q_amp
print(f"\n📌 PARAMETRE SAYISI:")
print(f"   Quantum (re-upload): {n_q_ru}")
print(f"   Quantum (amp)      : {n_q_amp}")
print(f"   Quantum toplam     : {n_q_total}")
print(f"   Classical          : {n_params - n_q_total}")
print(f"   GENEL TOPLAM       : {n_params}")

# ── Smoke Test ──────────────────────────────────────────────
print(f"\n📌 SMOKE TEST (batch=4, dual input):")
seedleri_sabitle(RANDOM_STATE)
test_x_pca = torch.tensor(QUANTUM_VERILER['X_train_pca6'][:4],
                           dtype=torch.float32)
test_x_amp = torch.tensor(QUANTUM_VERILER['X_train_amp16'][:4],
                           dtype=torch.float32)
test_logits = test_model(test_x_pca, test_x_amp)
print(f"   Output logits: {test_logits.shape}  (beklenen: torch.Size([4, 7]))")
if test_logits.shape == torch.Size([4, 7]):
    print(f"   ✅ Q3+ devre çalışıyor!")
else:
    raise ValueError(f"Shape hatalı: {test_logits.shape}")

# ── 5-Fold CV ──────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"🚀 5-FOLD CV BAŞLIYOR (~22-28 dk)")
print(f"{'='*65}")

q_q3plus_sonuc = quantum_5fold_cv(
    model_fabrika      = hybrid_q3plus_fabrika,
    model_adi          = MODEL_ADI_Q3P,
    X_train_full       = QUANTUM_VERILER['X_train_pca6'],
    y_train_full       = QUANTUM_VERILER['y_train'],
    X_test_full        = QUANTUM_VERILER['X_test_pca6'],
    y_test_full        = QUANTUM_VERILER['y_test'],
    X_train_2_full     = QUANTUM_VERILER['X_train_amp16'],
    X_test_2_full      = QUANTUM_VERILER['X_test_amp16'],
    n_epochs           = N_EPOCHS_Q3P,
    lr                 = LR_Q3P,
    batch_size         = BATCH_Q3P,
    patience           = PATIENCE_Q3P,
    use_class_weights  = True,
    model_aciklama     = "Q3-Plus: ReUpload-6q-2block + Amplitude-4q-2L"
)

# Sonuç değerlendirme
print(f"\n{'='*65}")
print(f"🎯 SONUÇ DEĞERLENDİRMESİ")
print(f"{'='*65}")
acc = q_q3plus_sonuc['ozet']['accuracy_mean']
q3_acc = 0.7711  # Q-Hybrid-Q3 sonucu

if acc >= 0.80:
    print(f"   ✅ ŞAMPİYON ADAYI: %{acc*100:.2f} (Q3'ün %{q3_acc*100:.2f}'üne göre +"
          f"{(acc-q3_acc)*100:.2f} puan)")
    print(f"   📌 Makaleye eklemeye değer — yeni şampiyon olabilir")
elif acc >= q3_acc:
    print(f"   📊 İYİLEŞME: %{acc*100:.2f} (Q3'ten +{(acc-q3_acc)*100:.2f} puan)")
    print(f"   📌 Marjinal kazanç, makaleye eklemek isteğe bağlı")
elif acc >= 0.70:
    print(f"   📊 BENZER: %{acc*100:.2f} (Q3 ile yakın)")
    print(f"   📌 Q3 sweet spot bulgusu — derinleştirme fayda etmiyor")
else:
    print(f"   📉 DÜŞÜK: %{acc*100:.2f} (Q3'ten geride)")
    print(f"   📌 Re-uploading + amplitude kombinasyonu çakıştı")


In [ ]:
# 6 ana model + Q3-Plus (toplam 7) konsolidasyonu, sıralama,
# görseller, klasik vs quantum karşılaştırma, ablation eğrisi

print("=" * 65)
print("📊 QUANTUM MODELLER — KONSOLİDE EDİLMİŞ ÖZET (7 MODEL)")
print("=" * 65)

# ── Adım 1: Tüm Quantum Sonuçları Tabloya Çevir ────────────
quantum_sonuclar = TUM_SONUCLAR['quantum']
print(f"\n📌 Toplam quantum model sonucu: {len(quantum_sonuclar)}")

q_satirlar = []
for s in quantum_sonuclar:
    q_satirlar.append({
        'Model'        : s['model'],
        'CV_Acc_Mean'  : s['accuracy_mean'],
        'CV_Acc_Std'   : s['accuracy_std'],
        'CV_F1_macro'  : s['f1_macro_mean'],
        'CV_F1_wgt'    : s['f1_weighted_mean'],
        'CV_MCC'       : s['mcc_mean'],
        'CV_Kappa'     : s['kappa_mean'],
        'CV_AUC_macro' : s.get('auc_macro_mean', None),
        'Sure_sn'      : s.get('sure_sn', '-'),
        'N_Epochs_Max' : s.get('n_epochs_max', '-'),
        'Mean_Best_Ep' : s.get('mean_best_epoch', '-'),
    })

quantum_df = pd.DataFrame(q_satirlar)
quantum_df_sirali = quantum_df.sort_values(
    by='CV_Acc_Mean', ascending=False
).reset_index(drop=True)
quantum_df_sirali.index += 1

print(f"\n📊 SIRALI ÖZET (CV Accuracy):")
print("=" * 105)
display_q = quantum_df_sirali[[
    'Model', 'CV_Acc_Mean', 'CV_Acc_Std',
    'CV_F1_macro', 'CV_MCC', 'CV_AUC_macro', 'Sure_sn'
]].copy()
for col in ['CV_Acc_Mean', 'CV_Acc_Std', 'CV_F1_macro', 'CV_MCC', 'CV_AUC_macro']:
    display_q[col] = display_q[col].apply(
        lambda x: f"{x:.4f}" if pd.notnull(x) else "-"
    )
print(display_q.to_string())

# ── Adım 2: Quantum Şampiyon ────────────────────────────────
q_sampiyon = quantum_df_sirali.iloc[0]
print(f"\n{'='*65}")
print(f"🏆 QUANTUM ŞAMPİYON")
print(f"{'='*65}")
print(f"   Model       : {q_sampiyon['Model']}")
print(f"   CV Accuracy : {q_sampiyon['CV_Acc_Mean']:.4f} ± {q_sampiyon['CV_Acc_Std']:.4f}")
print(f"   CV F1 macro : {q_sampiyon['CV_F1_macro']:.4f}")
print(f"   CV MCC      : {q_sampiyon['CV_MCC']:.4f}")
print(f"   CV AUC      : {q_sampiyon['CV_AUC_macro']:.4f}")

# ── Adım 3: Klasik vs Quantum Karşılaştırma ─────────────────
klasik_sampiyon = max(TUM_SONUCLAR['klasik'], key=lambda x: x['accuracy_mean'])

print(f"\n{'='*65}")
print(f"⚔️  KLASİK vs QUANTUM ŞAMPİYON KARŞILAŞTIRMASI")
print(f"{'='*65}")
print(f"   {'Metrik':<22} {'Klasik':>15} {'Quantum':>15} {'Fark':>12}")
print(f"   {'-'*22} {'-'*15} {'-'*15} {'-'*12}")
print(f"   {'Model':<22} "
      f"{klasik_sampiyon['model'][:15]:>15} "
      f"{q_sampiyon['Model'][:15]:>15} "
      f"{'':>12}")
print(f"   {'CV Accuracy':<22} "
      f"{klasik_sampiyon['accuracy_mean']:>15.4f} "
      f"{q_sampiyon['CV_Acc_Mean']:>15.4f} "
      f"{(q_sampiyon['CV_Acc_Mean']-klasik_sampiyon['accuracy_mean']):>+12.4f}")
print(f"   {'CV F1 (macro)':<22} "
      f"{klasik_sampiyon['f1_macro_mean']:>15.4f} "
      f"{q_sampiyon['CV_F1_macro']:>15.4f} "
      f"{(q_sampiyon['CV_F1_macro']-klasik_sampiyon['f1_macro_mean']):>+12.4f}")
print(f"   {'CV MCC':<22} "
      f"{klasik_sampiyon['mcc_mean']:>15.4f} "
      f"{q_sampiyon['CV_MCC']:>15.4f} "
      f"{(q_sampiyon['CV_MCC']-klasik_sampiyon['mcc_mean']):>+12.4f}")
print(f"   {'CV AUC (macro)':<22} "
      f"{klasik_sampiyon['auc_macro_mean']:>15.4f} "
      f"{q_sampiyon['CV_AUC_macro']:>15.4f} "
      f"{(q_sampiyon['CV_AUC_macro']-klasik_sampiyon['auc_macro_mean']):>+12.4f}")

gap = klasik_sampiyon['accuracy_mean'] - q_sampiyon['CV_Acc_Mean']
print(f"\n   📌 Klasik-Quantum Gap : {gap*100:.2f} puan accuracy")
print(f"   📌 AUC Gap           : {(klasik_sampiyon['auc_macro_mean']-q_sampiyon['CV_AUC_macro']):.4f}")

# ── Adım 4: Re-uploading Ablation Eğrisi ────────────────────
print(f"\n📊 GÖRSEL 1: Re-uploading Ablation Eğrisi")

ablation_data = {}
for s in quantum_sonuclar:
    if 'ReUpload' in s['model'] and 'block' in s['model']:
        if '1block' in s['model']:
            ablation_data[1] = (s['accuracy_mean'], s['accuracy_std'])
        elif '2block' in s['model']:
            ablation_data[2] = (s['accuracy_mean'], s['accuracy_std'])
        elif '3block' in s['model']:
            ablation_data[3] = (s['accuracy_mean'], s['accuracy_std'])

if len(ablation_data) == 3:
    fig, ax = plt.subplots(figsize=(9, 5.5))
    blocks = sorted(ablation_data.keys())
    means = [ablation_data[b][0] for b in blocks]
    stds  = [ablation_data[b][1] for b in blocks]

    ax.errorbar(blocks, means, yerr=stds, fmt='o-',
                linewidth=2.5, markersize=11,
                capsize=8, capthick=2, color='#e67e22',
                label='ReUpload pure (PCA-6)')

    # Klasik şampiyon referans çizgisi
    ax.axhline(y=klasik_sampiyon['accuracy_mean'],
               color='red', linestyle='--', alpha=0.7, linewidth=2,
               label=f"Klasik şampiyon ({klasik_sampiyon['model']}): "
                     f"{klasik_sampiyon['accuracy_mean']:.3f}")
    # Quantum şampiyon referans çizgisi
    ax.axhline(y=q_sampiyon['CV_Acc_Mean'],
               color='green', linestyle=':', alpha=0.7, linewidth=2,
               label=f"Quantum şampiyon ({q_sampiyon['Model']}): "
                     f"{q_sampiyon['CV_Acc_Mean']:.3f}")

    for b, m, s in zip(blocks, means, stds):
        ax.annotate(f'{m:.3f}\n±{s:.3f}', (b, m),
                    textcoords="offset points", xytext=(12, 5),
                    fontsize=9, fontweight='bold')

    ax.set_xlabel('Re-uploading Block Sayısı', fontweight='bold', fontsize=11)
    ax.set_ylabel('CV Accuracy (Mean ± Std)', fontweight='bold', fontsize=11)
    ax.set_title('Re-uploading Depth Ablation\n(Obezite, 5-Fold CV, Multiclass-7)',
                 fontweight='bold', fontsize=12)
    ax.set_xticks(blocks)
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0.4, 1.0])

    plt.tight_layout()
    abl_yol = f'{GORSELLER}/quantum_reupload_ablation.png'
    plt.savefig(abl_yol, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Kaydedildi: {abl_yol}")

# ── Adım 5: Tüm Quantum Modeller Bar Chart ──────────────────
print(f"\n📊 GÖRSEL 2: Tüm Quantum Modeller Karşılaştırması")

fig, ax = plt.subplots(figsize=(12, 6))
df_q_plot = quantum_df.sort_values('CV_Acc_Mean', ascending=True)

renkler = []
for m in df_q_plot['Model']:
    if 'Q3Plus' in m:
        renkler.append('#16a085')  # koyu yeşil — yeni şampiyon
    elif 'Q3' in m and 'Plus' not in m:
        renkler.append('#27ae60')  # yeşil — eski şampiyon
    elif 'Amplitude' in m:
        renkler.append('#9b59b6')  # mor
    elif 'Angle' in m:
        renkler.append('#3498db')  # mavi
    else:  # ReUpload
        renkler.append('#e67e22')  # turuncu

bars = ax.barh(df_q_plot['Model'], df_q_plot['CV_Acc_Mean'],
               xerr=df_q_plot['CV_Acc_Std'],
               color=renkler, edgecolor='black', alpha=0.85,
               error_kw={'capsize': 4, 'capthick': 1.5})

# Klasik şampiyon referans çizgisi
ax.axvline(x=klasik_sampiyon['accuracy_mean'],
           color='red', linestyle='--', alpha=0.7,
           linewidth=2, label=f"Klasik şampiyon: "
                              f"{klasik_sampiyon['accuracy_mean']:.3f}")

for bar, v in zip(bars, df_q_plot['CV_Acc_Mean']):
    ax.text(v + 0.005, bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=9, fontweight='bold')

ax.set_xlabel('CV Accuracy (Mean ± Std)', fontweight='bold', fontsize=11)
ax.set_title('Quantum Modeller — Karşılaştırmalı Performans (5-Fold CV)\n'
             'Obezite Veri Seti, Multiclass-7',
             fontweight='bold', fontsize=12)
ax.set_xlim([0.4, 1.0])
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
qbar_yol = f'{GORSELLER}/quantum_karsilastirma_bar.png'
plt.savefig(qbar_yol, dpi=150, bbox_inches='tight')
plt.show()
print(f"   💾 Kaydedildi: {qbar_yol}")

# ── Adım 6: Klasik vs Quantum Birleşik ──────────────────────
print(f"\n📊 GÖRSEL 3: Klasik vs Quantum (Birleşik)")

# Top 5 klasik (top-10 versiyonları öncelikli)
top5_klasik = sorted(TUM_SONUCLAR['klasik'],
                     key=lambda x: x['accuracy_mean'], reverse=True)[:5]

fig, ax = plt.subplots(figsize=(13, 8))

# Klasik bar'lar
klasik_isimler = [s['model'] for s in top5_klasik]
klasik_meanler = [s['accuracy_mean'] for s in top5_klasik]
klasik_stdler  = [s['accuracy_std']  for s in top5_klasik]

# Quantum bar'lar
quantum_isimler = list(df_q_plot['Model'])
quantum_meanler = list(df_q_plot['CV_Acc_Mean'])
quantum_stdler  = list(df_q_plot['CV_Acc_Std'])

# Birleşik
tum_isimler = klasik_isimler + quantum_isimler
tum_meanler = klasik_meanler + quantum_meanler
tum_stdler  = klasik_stdler  + quantum_stdler
tum_renkler = ['#34495e'] * 5 + renkler

# Sırala
sirali_idx = np.argsort(tum_meanler)
tum_isimler_s = [tum_isimler[i] for i in sirali_idx]
tum_meanler_s = [tum_meanler[i] for i in sirali_idx]
tum_stdler_s  = [tum_stdler[i] for i in sirali_idx]
tum_renkler_s = [tum_renkler[i] for i in sirali_idx]

bars = ax.barh(tum_isimler_s, tum_meanler_s, xerr=tum_stdler_s,
               color=tum_renkler_s, edgecolor='black', alpha=0.85,
               error_kw={'capsize': 4})

for bar, v in zip(bars, tum_meanler_s):
    ax.text(v + 0.005, bar.get_y() + bar.get_height()/2,
            f'{v:.3f}', va='center', fontsize=9)

ax.set_xlabel('CV Accuracy (Mean ± Std)', fontweight='bold', fontsize=11)
ax.set_title('Klasik vs Quantum — Tüm Modeller Karşılaştırması\n'
             'Obezite Veri Seti, 5-Fold Stratified CV',
             fontweight='bold', fontsize=12)
ax.set_xlim([0.4, 1.0])
ax.grid(axis='x', alpha=0.3)

# Legend manuel
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#34495e', edgecolor='black', label='Klasik (top-5)'),
    Patch(facecolor='#16a085', edgecolor='black', label='Quantum: Q3-Plus (Şampiyon)'),
    Patch(facecolor='#27ae60', edgecolor='black', label='Quantum: Q3 Hybrid'),
    Patch(facecolor='#9b59b6', edgecolor='black', label='Quantum: Amplitude'),
    Patch(facecolor='#3498db', edgecolor='black', label='Quantum: Angle'),
    Patch(facecolor='#e67e22', edgecolor='black', label='Quantum: ReUpload'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
karsi_yol = f'{GORSELLER}/klasik_vs_quantum_birlesik.png'
plt.savefig(karsi_yol, dpi=150, bbox_inches='tight')
plt.show()
print(f"   💾 Kaydedildi: {karsi_yol}")

# ── Adım 7: Encoding Aile Karşılaştırması ──────────────────
print(f"\n📊 GÖRSEL 4: Encoding Aile Bazında Karşılaştırma")

# Aile gruplaması
aile_gruplama = {
    'Pure Angle'        : [s for s in quantum_sonuclar
                           if 'Angle' in s['model'] and 'Plus' not in s['model']],
    'Pure Amplitude'    : [s for s in quantum_sonuclar
                           if 'Amplitude' in s['model']
                           and 'Plus' not in s['model']],
    'Re-uploading\n(en iyi)' : [max([s for s in quantum_sonuclar
                                  if 'ReUpload' in s['model']
                                  and 'block' in s['model']],
                                 key=lambda x: x['accuracy_mean'])]
                                if any('ReUpload' in s['model']
                                       and 'block' in s['model']
                                       for s in quantum_sonuclar) else [],
    'Hybrid Q3'         : [s for s in quantum_sonuclar
                           if 'Q3' in s['model']
                           and 'Plus' not in s['model']],
    'Hybrid Q3-Plus\n(Şampiyon)': [s for s in quantum_sonuclar
                                  if 'Q3Plus' in s['model']],
}

# En iyi her aileden
aile_isim = []
aile_acc  = []
aile_std  = []
for aile, modeller in aile_gruplama.items():
    if modeller:
        en_iyi = max(modeller, key=lambda x: x['accuracy_mean'])
        aile_isim.append(aile)
        aile_acc.append(en_iyi['accuracy_mean'])
        aile_std.append(en_iyi['accuracy_std'])

fig, ax = plt.subplots(figsize=(11, 5.5))
renkler_aile = ['#3498db', '#9b59b6', '#e67e22', '#27ae60', '#16a085']

bars = ax.bar(aile_isim, aile_acc, yerr=aile_std,
              color=renkler_aile[:len(aile_isim)],
              edgecolor='black', alpha=0.85,
              capsize=8, error_kw={'capthick': 1.5})

for bar, v in zip(bars, aile_acc):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.015,
            f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')

ax.axhline(y=klasik_sampiyon['accuracy_mean'],
           color='red', linestyle='--', alpha=0.7, linewidth=2,
           label=f"Klasik şampiyon: {klasik_sampiyon['accuracy_mean']:.3f}")

ax.set_ylabel('CV Accuracy (Mean ± Std)', fontweight='bold', fontsize=11)
ax.set_title('Quantum Encoding Aile Karşılaştırması\n'
             '(Her aileden en iyi model)',
             fontweight='bold', fontsize=12)
ax.set_ylim([0.4, 1.0])
ax.legend(loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
aile_yol = f'{GORSELLER}/quantum_aile_karsilastirma.png'
plt.savefig(aile_yol, dpi=150, bbox_inches='tight')
plt.show()
print(f"   💾 Kaydedildi: {aile_yol}")

# ── Adım 8: CSV ve JSON Kaydet ──────────────────────────────
csv_yol = f'{OUTPUT_DIR}/sonuclar/QUANTUM_OZET_TABLO.csv'
quantum_df_sirali.to_csv(csv_yol, index=False)
print(f"\n💾 Tablo CSV kaydedildi: {csv_yol}")

konsolidasyon = {
    "tarih": datetime.now().strftime('%d/%m/%Y %H:%M:%S'),
    "n_quantum_model": len(quantum_sonuclar),
    "quantum_sampiyon": {
        "model"       : q_sampiyon['Model'],
        "cv_acc_mean" : float(q_sampiyon['CV_Acc_Mean']),
        "cv_acc_std"  : float(q_sampiyon['CV_Acc_Std']),
        "cv_f1_macro" : float(q_sampiyon['CV_F1_macro']),
        "cv_mcc"      : float(q_sampiyon['CV_MCC']),
        "cv_auc_macro": float(q_sampiyon['CV_AUC_macro']),
    },
    "klasik_sampiyon": {
        "model"       : klasik_sampiyon['model'],
        "cv_acc_mean" : float(klasik_sampiyon['accuracy_mean']),
        "cv_f1_macro" : float(klasik_sampiyon['f1_macro_mean']),
        "cv_auc_macro": float(klasik_sampiyon['auc_macro_mean']),
    },
    "gap_analizi": {
        "accuracy_gap" : float(klasik_sampiyon['accuracy_mean'] - q_sampiyon['CV_Acc_Mean']),
        "f1_macro_gap" : float(klasik_sampiyon['f1_macro_mean'] - q_sampiyon['CV_F1_macro']),
        "auc_gap"      : float(klasik_sampiyon['auc_macro_mean'] - q_sampiyon['CV_AUC_macro']),
    },
    "ablation_reupload": {
        f"{b}_block": {"acc_mean": float(d[0]), "acc_std": float(d[1])}
        for b, d in ablation_data.items()
    },
    "encoding_aile_ozet": {
        aile: {"acc_mean": float(acc), "acc_std": float(std)}
        for aile, acc, std in zip(aile_isim, aile_acc, aile_std)
    },
    "tum_quantum_sonuclar": quantum_sonuclar
}

konsol_yol = f'{OUTPUT_DIR}/sonuclar/QUANTUM_KONSOLIDASYON.json'
with open(konsol_yol, 'w', encoding='utf-8') as f:
    json.dump(konsolidasyon, f, ensure_ascii=False, indent=4, default=str)
print(f"💾 Konsolidasyon JSON: {konsol_yol}")

# ── Final Özet ──────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"{'='*65}")
print(f"   Toplam quantum model : {len(quantum_sonuclar)}")
print(f"   Quantum şampiyon     : {q_sampiyon['Model']}")
print(f"   Quantum CV Acc       : {q_sampiyon['CV_Acc_Mean']:.4f} ± "
      f"{q_sampiyon['CV_Acc_Std']:.4f}")
print(f"   Quantum AUC          : {q_sampiyon['CV_AUC_macro']:.4f}")
print(f"   Klasik şampiyon      : {klasik_sampiyon['model']}")
print(f"   Klasik CV Acc        : {klasik_sampiyon['accuracy_mean']:.4f}")
print(f"   Acc Gap              : {gap*100:.2f} puan")

print(f"   - Hücre 24: Noise robustness ({q_sampiyon['Model']})")
print(f"   - Hücre 25: SHAP analizi (XGBoost)")
print(f"   - Hücre 26: Paired Wilcoxon istatistik")


In [ ]:
# Depolarizing + Bit-flip noise, 4 nokta her biri için

print("=" * 65)
print("🌀 NOISE ROBUSTNESS ANALİZİ — Q-Hybrid-Q3-Plus")
print("=" * 65)

# ── Noise Sabitleri ─────────────────────────────────────────
NOISE_LEVELS = [0.0, 0.05, 0.10, 0.15, 0.20]
NOISE_TYPES  = ['depolarizing', 'bit_flip']

# ── Noise'lu Devre Tanımları ────────────────────────────────
# Branch 1 (Re-Upload, 6 qubit) — noise'lu versiyonu
def make_noisy_reupload_branch(noise_type, noise_p):
    """Belirtilen noise ile re-upload branch'i üret."""
    dev = qml.device('default.mixed', wires=RU_QUBITS_Q3P)

    @qml.qnode(dev, interface='torch', diff_method='backprop')
    def noisy_ru_branch(inputs, weights):
        for block in range(RU_BLOCKS_Q3P):
            for i in range(RU_QUBITS_Q3P):
                qml.RY(inputs[..., i], wires=i)
                if noise_p > 0:
                    if noise_type == 'depolarizing':
                        qml.DepolarizingChannel(noise_p, wires=i)
                    elif noise_type == 'bit_flip':
                        qml.BitFlip(noise_p, wires=i)
            for i in range(RU_QUBITS_Q3P):
                qml.Rot(weights[block, i, 0], weights[block, i, 1],
                        weights[block, i, 2], wires=i)
                if noise_p > 0:
                    if noise_type == 'depolarizing':
                        qml.DepolarizingChannel(noise_p, wires=i)
                    elif noise_type == 'bit_flip':
                        qml.BitFlip(noise_p, wires=i)
            for i in range(RU_QUBITS_Q3P):
                qml.CNOT(wires=[i, (i + 1) % RU_QUBITS_Q3P])

        return [qml.expval(qml.PauliZ(i)) for i in range(RU_QUBITS_Q3P)]

    return noisy_ru_branch

def make_noisy_amp_branch(noise_type, noise_p):
    """Belirtilen noise ile amplitude branch'i üret."""
    dev = qml.device('default.mixed', wires=AMP_QUBITS_Q3P)

    @qml.qnode(dev, interface='torch', diff_method='backprop')
    def noisy_amp_branch(inputs, weights):
        qml.AmplitudeEmbedding(inputs, wires=range(AMP_QUBITS_Q3P), normalize=True)
        if noise_p > 0:
            for i in range(AMP_QUBITS_Q3P):
                if noise_type == 'depolarizing':
                    qml.DepolarizingChannel(noise_p, wires=i)
                elif noise_type == 'bit_flip':
                    qml.BitFlip(noise_p, wires=i)

        for layer in range(AMP_LAYERS_Q3P):
            for i in range(AMP_QUBITS_Q3P):
                qml.RY(weights[layer, i, 0], wires=i)
                qml.RZ(weights[layer, i, 1], wires=i)
                if noise_p > 0:
                    if noise_type == 'depolarizing':
                        qml.DepolarizingChannel(noise_p, wires=i)
                    elif noise_type == 'bit_flip':
                        qml.BitFlip(noise_p, wires=i)
            for i in range(AMP_QUBITS_Q3P):
                qml.CNOT(wires=[i, (i + 1) % AMP_QUBITS_Q3P])

        return [qml.expval(qml.PauliZ(i)) for i in range(AMP_QUBITS_Q3P)]

    return noisy_amp_branch

def make_noisy_q3plus_model(noise_type, noise_p):
    """Tüm Q3-Plus modelini noise'lu yapar."""
    noisy_ru = make_noisy_reupload_branch(noise_type, noise_p)
    noisy_amp = make_noisy_amp_branch(noise_type, noise_p)

    class NoisyQ3PlusModel(nn.Module):
        def __init__(self):
            super().__init__()
            ru_shapes = {"weights": (RU_BLOCKS_Q3P, RU_QUBITS_Q3P, 3)}
            amp_shapes = {"weights": (AMP_LAYERS_Q3P, AMP_QUBITS_Q3P, 2)}
            self.ru_qlayer = qml.qnn.TorchLayer(noisy_ru, ru_shapes)
            self.amp_qlayer = qml.qnn.TorchLayer(noisy_amp, amp_shapes)
            n_concat = RU_QUBITS_Q3P + AMP_QUBITS_Q3P
            self.classical_head = nn.Sequential(
                nn.Linear(n_concat, N_HIDDEN_Q3P),
                nn.ReLU(),
                nn.Dropout(DROPOUT_Q3P),
                nn.Linear(N_HIDDEN_Q3P, N_CLASSES)
            )

        def forward(self, x_pca, x_amp):
            pca_single = (x_pca.dim() == 1)
            amp_single = (x_amp.dim() == 1)
            if pca_single:
                x_pca = x_pca.unsqueeze(0)
            if amp_single:
                x_amp = x_amp.unsqueeze(0)
            ru_out  = self.ru_qlayer(x_pca)
            amp_out = self.amp_qlayer(x_amp)
            fused   = torch.cat([ru_out, amp_out], dim=1)
            logits  = self.classical_head(fused)
            if pca_single and amp_single:
                logits = logits.squeeze(0)
            return logits

    return NoisyQ3PlusModel

# ── Önce Temiz (Noise Yok) Şampiyon Modeli Yeniden Eğit ─────
print(f"\n📌 ADIM 1: Temiz şampiyon modeli yeniden eğitiliyor (referans)...")
seedleri_sabitle(RANDOM_STATE + 200)

noise_baseline_model = hybrid_q3plus_fabrika()
nb_optim = optim.Adam(noise_baseline_model.parameters(), lr=LR_Q3P)
cw = class_weights_hesapla(QUANTUM_VERILER['y_train'])
nb_crit = nn.CrossEntropyLoss(weight=cw)

X_tr_pca_t = torch.tensor(QUANTUM_VERILER['X_train_pca6'], dtype=torch.float32)
X_te_pca_t = torch.tensor(QUANTUM_VERILER['X_test_pca6'],  dtype=torch.float32)
X_tr_amp_t = torch.tensor(QUANTUM_VERILER['X_train_amp16'], dtype=torch.float32)
X_te_amp_t = torch.tensor(QUANTUM_VERILER['X_test_amp16'],  dtype=torch.float32)
y_tr_t     = torch.tensor(QUANTUM_VERILER['y_train'], dtype=torch.long)
y_te       = QUANTUM_VERILER['y_test']

ds = TensorDataset(X_tr_pca_t, X_tr_amp_t, y_tr_t)
loader = DataLoader(ds, batch_size=32, shuffle=True)

# 50 epoch ile retrain (ortalama best_epoch civarı)
N_RETRAIN_EP = 50
print(f"   Retrain epochs: {N_RETRAIN_EP}")
for epoch in range(N_RETRAIN_EP):
    noise_baseline_model.train()
    for batch in loader:
        nb_optim.zero_grad()
        xb, xb2, yb = batch
        logits = noise_baseline_model(xb, xb2)
        loss = nb_crit(logits, yb)
        loss.backward()
        nb_optim.step()

# Temiz state'i kaydet (noise modellerine yükleyeceğiz)
clean_state = {k: v.clone() for k, v in noise_baseline_model.state_dict().items()}

# Temiz baseline test
noise_baseline_model.eval()
with torch.no_grad():
    clean_logits = noise_baseline_model(X_te_pca_t, X_te_amp_t)
    clean_preds = torch.argmax(clean_logits, dim=1).numpy()
    clean_acc = accuracy_score(y_te, clean_preds)

print(f"   ✅ Temiz (noise=0) Test Acc: {clean_acc:.4f}")

# ── Noise Spektrumu Tarama ──────────────────────────────────
print(f"\n📌 ADIM 2: Noise spektrumu taraması başlıyor...")
print(f"   Noise türleri : {NOISE_TYPES}")
print(f"   Noise seviyeleri: {NOISE_LEVELS}")
print(f"   Toplam nokta  : {len(NOISE_TYPES) * (len(NOISE_LEVELS)-1)} + 1 (clean)")

noise_sonuclar = {nt: {'levels': [], 'accs': []} for nt in NOISE_TYPES}

for nt in NOISE_TYPES:
    print(f"\n   🌀 Noise türü: {nt}")
    for p in NOISE_LEVELS:
        if p == 0.0:
            # Clean baseline
            acc = clean_acc
            print(f"     p={p:.2f}: Test Acc = {acc:.4f} (clean baseline)")
        else:
            # Noise'lu model oluştur, temiz state'i yükle
            ModelClass = make_noisy_q3plus_model(nt, p)
            noisy_model = ModelClass()
            try:
                noisy_model.load_state_dict(clean_state, strict=False)
            except Exception as e:
                print(f"     p={p:.2f}: state load uyarısı — {e}")

            # Test seti üzerinde değerlendir
            noisy_model.eval()
            with torch.no_grad():
                logits = noisy_model(X_te_pca_t, X_te_amp_t)
                preds = torch.argmax(logits, dim=1).numpy()
                acc = accuracy_score(y_te, preds)

            print(f"     p={p:.2f}: Test Acc = {acc:.4f} "
                  f"(Δ vs clean: {(acc-clean_acc)*100:+.2f}%)")

        noise_sonuclar[nt]['levels'].append(p)
        noise_sonuclar[nt]['accs'].append(acc)

# ── Görselleştirme ──────────────────────────────────────────
print(f"\n📊 GÖRSEL: Noise Robustness Spektrumu")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, nt in zip(axes, NOISE_TYPES):
    levels = noise_sonuclar[nt]['levels']
    accs   = noise_sonuclar[nt]['accs']

    ax.plot(levels, accs, 'o-', linewidth=2.5, markersize=10,
            color='#16a085', label='Q-Hybrid-Q3-Plus')
    ax.axhline(y=clean_acc, color='green', linestyle=':', alpha=0.6,
               label=f'Clean baseline: {clean_acc:.3f}')
    ax.axhline(y=0.143, color='gray', linestyle='--', alpha=0.5,
               label='Random (1/7 = 0.143)')

    for x, y in zip(levels, accs):
        ax.annotate(f'{y:.3f}', (x, y),
                    textcoords="offset points", xytext=(0, 10),
                    fontsize=9, ha='center', fontweight='bold')

    ax.set_xlabel(f'{nt.capitalize()} Noise Probability (p)',
                  fontweight='bold')
    ax.set_ylabel('Test Accuracy', fontweight='bold')
    ax.set_title(f'{nt.capitalize()} Noise Robustness',
                 fontweight='bold')
    ax.set_ylim([0.0, 1.0])
    ax.legend(loc='lower left', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Q-Hybrid-Q3-Plus Noise Robustness Analizi (Obezite)',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
noise_yol = f'{GORSELLER}/quantum_noise_spectrum.png'
plt.savefig(noise_yol, dpi=150, bbox_inches='tight')
plt.show()
print(f"   💾 Kaydedildi: {noise_yol}")

# ── Kayıt ──────────────────────────────────────────────────
noise_kayit = {
    "tarih"      : datetime.now().strftime('%d/%m/%Y %H:%M:%S'),
    "champion"   : "Q_Hybrid_Q3Plus_ReUpAmp",
    "clean_acc"  : float(clean_acc),
    "noise_levels": NOISE_LEVELS,
    "noise_types": NOISE_TYPES,
    "results": {
        nt: {
            "levels": [float(l) for l in noise_sonuclar[nt]['levels']],
            "accs"  : [float(a) for a in noise_sonuclar[nt]['accs']]
        }
        for nt in NOISE_TYPES
    }
}

noise_yol_json = f'{OUTPUT_DIR}/sonuclar/quantum_noise_spectrum.json'
with open(noise_yol_json, 'w', encoding='utf-8') as f:
    json.dump(noise_kayit, f, ensure_ascii=False, indent=4, default=str)
print(f"💾 JSON kaydedildi: {noise_yol_json}")

# ── Özet ───────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"📊 NOİSE ROBUSTNESS ÖZETİ")
print(f"{'='*65}")
print(f"   Temiz baseline: Test Acc = {clean_acc:.4f}")
print(f"\n   Depolarizing noise (en şiddetli p=0.20):")
print(f"     Test Acc = {noise_sonuclar['depolarizing']['accs'][-1]:.4f}")
print(f"     Düşüş   = {(clean_acc - noise_sonuclar['depolarizing']['accs'][-1])*100:.2f}%")
print(f"\n   Bit-flip noise (en şiddetli p=0.20):")
print(f"     Test Acc = {noise_sonuclar['bit_flip']['accs'][-1]:.4f}")
print(f"     Düşüş   = {(clean_acc - noise_sonuclar['bit_flip']['accs'][-1])*100:.2f}%")


In [ ]:
# Klasik şampiyon üzerinde klinik feature importance

print("=" * 65)
print("🔍 SHAP ANALİZİ — XGBoost (Klasik Şampiyon)")
print("=" * 65)

try:
    import shap
    print("✅ SHAP yüklü")
except ImportError:
    print("📦 SHAP yükleniyor...")
    import subprocess
    subprocess.check_call(['pip', 'install', '--quiet', 'shap'])
    import shap
    print("✅ SHAP yüklendi")

# ── Adım 1: Verileri Preprocessor ile Hazırla ──────────────
print(f"\n📌 ADIM 1: Top-10 verisi preprocessor ile hazırlanıyor...")

from xgboost import XGBClassifier

top10_features = KLASIK_VERILER['top10_features']
print(f"   Top-10 features: {top10_features}")

# DataFrame'leri al
X_train_df = KLASIK_VERILER['X_train_10']
X_test_df  = KLASIK_VERILER['X_test_10']
y_train    = KLASIK_VERILER['y_train']
y_test     = KLASIK_VERILER['y_test']
preprocessor_10 = KLASIK_VERILER['preprocessor_10']

# Preprocessor'ı eğit (train) ve uygula (train+test)
print(f"   Preprocessor fit + transform ediliyor...")
X_train_proc = preprocessor_10.fit_transform(X_train_df)
X_test_proc  = preprocessor_10.transform(X_test_df)

# Sparse ise dense'e çevir
if hasattr(X_train_proc, 'toarray'):
    X_train_proc = X_train_proc.toarray()
    X_test_proc  = X_test_proc.toarray()

print(f"   X_train shape : {X_train_proc.shape}")
print(f"   X_test shape  : {X_test_proc.shape}")

# Preprocessor'dan gelen gerçek feature isimleri
try:
    feature_names_out = preprocessor_10.get_feature_names_out().tolist()
    # Prefix temizle ('num__', 'cat__' gibi)
    feature_names_clean = [f.split('__')[-1] for f in feature_names_out]
    print(f"   Preprocessed feature sayısı: {len(feature_names_clean)}")
    print(f"   İlk 5 feature: {feature_names_clean[:5]}")
except Exception as e:
    feature_names_clean = [f"f{i}" for i in range(X_train_proc.shape[1])]
    print(f"   ⚠️ Feature isim alınamadı, generic: {e}")

# ── Adım 2: XGBoost şampiyon modeli yeniden eğit ───────────
print(f"\n📌 ADIM 2: XGBoost top-10 modeli yeniden eğitiliyor...")

xgb_best_params = {
    'learning_rate': 0.05,
    'max_depth'    : 6,
    'n_estimators' : 200,
    'random_state' : RANDOM_STATE,
    'eval_metric'  : 'mlogloss',
}

shap_xgb = XGBClassifier(**xgb_best_params)
shap_xgb.fit(X_train_proc, y_train)

test_preds = shap_xgb.predict(X_test_proc)
test_acc = accuracy_score(y_test, test_preds)
print(f"   ✅ Test Accuracy : {test_acc:.4f}")

# ── Adım 3: SHAP Explainer ──────────────────────────────────
print(f"\n📌 ADIM 3: SHAP TreeExplainer hesaplanıyor...")
explainer = shap.TreeExplainer(shap_xgb)
shap_values = explainer.shap_values(X_test_proc)

if isinstance(shap_values, list):
    print(f"   SHAP values: List of {len(shap_values)} arrays, each {shap_values[0].shape}")
elif shap_values.ndim == 3:
    print(f"   SHAP values shape (3D): {shap_values.shape}")
else:
    print(f"   SHAP values shape: {shap_values.shape}")

sinif_isimleri = list(KLASIK_VERILER['label_encoder'].classes_)
print(f"   Sınıflar: {sinif_isimleri}")

# ── Adım 4: Global Feature Importance ──────────────────────
print(f"\n📊 GÖRSEL 1: Global Feature Importance (Mean |SHAP|)")

if isinstance(shap_values, list):
    mean_abs_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
elif shap_values.ndim == 3:
    mean_abs_shap = np.abs(shap_values).mean(axis=(0, 2))
else:
    mean_abs_shap = np.abs(shap_values).mean(axis=0)

# Aynı ham feature'a ait one-hot sütunları topla
print(f"   📌 One-hot feature'ları toplama (gerçek isimleri için)...")
ham_feature_importance = {}
for i, fn in enumerate(feature_names_clean):
    # Kategorik feature'lar için 'Gender_Male' → 'Gender' gibi parçala
    ham_isim = fn
    for orijinal in top10_features:
        if fn.startswith(orijinal):
            ham_isim = orijinal
            break
    ham_feature_importance[ham_isim] = ham_feature_importance.get(ham_isim, 0) + mean_abs_shap[i]

# Sırala
sorted_items = sorted(ham_feature_importance.items(), key=lambda x: x[1], reverse=True)
sorted_features = [k for k, v in sorted_items]
sorted_importance = np.array([v for k, v in sorted_items])

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(sorted_features))
bars = ax.barh(y_pos, sorted_importance[::-1],
                color='#3498db', edgecolor='black', alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels(sorted_features[::-1], fontsize=10)

for bar, v in zip(bars, sorted_importance[::-1]):
    ax.text(v + max(sorted_importance) * 0.01,
            bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Mean |SHAP value| (sınıflar arası ortalama)',
              fontweight='bold', fontsize=11)
ax.set_title('XGBoost Top-10 — Global Feature Importance (SHAP)\n'
             'Obezite, Multiclass-7',
             fontweight='bold', fontsize=12)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
shap_global_yol = f'{GORSELLER}/shap_xgboost_global.png'
plt.savefig(shap_global_yol, dpi=150, bbox_inches='tight')
plt.show()
print(f"   💾 Kaydedildi: {shap_global_yol}")

# ── Adım 5: Klinik Yorum Tablosu ───────────────────────────
print(f"\n📊 KLİNİK YORUM TABLOSU:")
print("=" * 90)
klinik_yorum = {
    'Weight'                       : 'Mutlak ağırlık — BMI ana bileşeni',
    'Age'                          : 'Yaş — metabolik faktör',
    'FCVC'                         : 'Sebze tüketim sıklığı',
    'Height'                       : 'Boy — BMI normalleştirmesi',
    'Gender'                       : 'Cinsiyet — vücut kompozisyonu farkı',
    'NCP'                          : 'Günlük öğün sayısı',
    'CALC'                         : 'Alkol tüketimi',
    'FAF'                          : 'Fiziksel aktivite sıklığı',
    'TUE'                          : 'Teknoloji kullanım süresi (sedanter)',
    'family_history_with_overweight': 'Aile öyküsü — genetik yatkınlık',
}

print(f"   {'Sıra':<5} {'Feature':<35} {'Mean|SHAP|':<12} Klinik Anlamı")
print(f"   {'-'*5} {'-'*35} {'-'*12} {'-'*40}")
for i, (feat, imp) in enumerate(zip(sorted_features, sorted_importance), 1):
    aciklama = klinik_yorum.get(feat, '-')
    print(f"   {i:<5} {feat[:34]:<35} {imp:<12.4f} {aciklama}")

# ── Adım 6: SHAP Summary Plot (Beeswarm) ────────────────────
print(f"\n📊 GÖRSEL 2: SHAP Summary Plot — Obesity_Type_III")

# En iyi sınıflanmış sınıf: Obesity_Type_III (confusion matrix'te 65/65)
hedef_sinif_idx = sinif_isimleri.index('Obesity_Type_III')
print(f"   Sınıf: {sinif_isimleri[hedef_sinif_idx]} (idx={hedef_sinif_idx})")

if isinstance(shap_values, list):
    sv_to_plot = shap_values[hedef_sinif_idx]
elif shap_values.ndim == 3:
    sv_to_plot = shap_values[:, :, hedef_sinif_idx]
else:
    sv_to_plot = shap_values

plt.figure(figsize=(10, 7))
shap.summary_plot(
    sv_to_plot,
    X_test_proc,
    feature_names=feature_names_clean,
    show=False,
    max_display=15
)
plt.title(f'SHAP Summary — {sinif_isimleri[hedef_sinif_idx]}',
          fontweight='bold', fontsize=12)
plt.tight_layout()
shap_beeswarm_yol = f'{GORSELLER}/shap_xgboost_summary.png'
plt.savefig(shap_beeswarm_yol, dpi=150, bbox_inches='tight')
plt.show()
print(f"   💾 Kaydedildi: {shap_beeswarm_yol}")

# ── Adım 7: Sınıf-Bazlı SHAP Heatmap ────────────────────────
print(f"\n📊 GÖRSEL 3: Sınıf-Bazlı SHAP Importance Heatmap")

n_classes = len(sinif_isimleri)

# Her sınıf için, her ham feature'ın mean|SHAP|'ini topla
shap_class_matrix = np.zeros((n_classes, len(sorted_features)))

for c in range(n_classes):
    if isinstance(shap_values, list):
        sv_c = shap_values[c]
    elif shap_values.ndim == 3:
        sv_c = shap_values[:, :, c]
    else:
        continue

    sv_c_mean = np.abs(sv_c).mean(axis=0)

    # Ham feature'a topla
    ham_class = {}
    for i, fn in enumerate(feature_names_clean):
        ham_isim = fn
        for orijinal in top10_features:
            if fn.startswith(orijinal):
                ham_isim = orijinal
                break
        ham_class[ham_isim] = ham_class.get(ham_isim, 0) + sv_c_mean[i]

    for fi, fname in enumerate(sorted_features):
        shap_class_matrix[c, fi] = ham_class.get(fname, 0)

# Her satırı normalize
shap_class_norm = shap_class_matrix / (shap_class_matrix.max(axis=1, keepdims=True) + 1e-9)

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(shap_class_norm, annot=True, fmt='.2f',
            xticklabels=sorted_features, yticklabels=sinif_isimleri,
            cmap='YlOrRd', cbar_kws={'label': 'Normalized |SHAP|'},
            ax=ax)
ax.set_title('Sınıf-Bazlı Feature Importance (Normalized SHAP)\n'
             'Her satır kendi içinde 0-1 arası normalize edilmiştir',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Features', fontweight='bold')
ax.set_ylabel('Sınıflar', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
shap_heatmap_yol = f'{GORSELLER}/shap_xgboost_class_heatmap.png'
plt.savefig(shap_heatmap_yol, dpi=150, bbox_inches='tight')
plt.show()
print(f"   💾 Kaydedildi: {shap_heatmap_yol}")

# ── Adım 8: JSON Kaydet ─────────────────────────────────────
shap_kayit = {
    "tarih"        : datetime.now().strftime('%d/%m/%Y %H:%M:%S'),
    "model"        : "XGBoost_top10",
    "test_acc"     : float(test_acc),
    "global_importance": {
        feat: float(imp)
        for feat, imp in zip(sorted_features, sorted_importance)
    },
    "siniflar"     : sinif_isimleri,
    "class_importance": {
        sinif_isimleri[c]: {
            sorted_features[f]: float(shap_class_matrix[c, f])
            for f in range(len(sorted_features))
        }
        for c in range(n_classes)
    }
}

shap_yol_json = f'{OUTPUT_DIR}/sonuclar/shap_xgboost.json'
with open(shap_yol_json, 'w', encoding='utf-8') as f:
    json.dump(shap_kayit, f, ensure_ascii=False, indent=4, default=str)
print(f"\n💾 JSON kaydedildi: {shap_yol_json}")

# ── Final Özet ──────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"{'='*65}")
print(f"   Model              : XGBoost top-10")
print(f"   Test Accuracy      : {test_acc:.4f}")
print(f"   En önemli 3 feature:")
for i in range(3):
    print(f"     {i+1}. {sorted_features[i]} ({sorted_importance[i]:.4f})")


In [ ]:
# Klasik (XGBoost top-10) vs Quantum (Q3-Plus) fold-bazlı paired test

print("=" * 65)
print("📐 PAIRED WILCOXON SIGNED-RANK TEST")
print("=" * 65)

from scipy import stats
import os, glob

# ── Adım 1: Fold sonuçlarını topla ─────────────────────────
print(f"\n📌 ADIM 1: Fold-bazlı sonuçlar toplanıyor...")

# Klasik şampiyon kaydı (TUM_SONUCLAR'dan)
klasik_sampiyon_kayit = max(TUM_SONUCLAR['klasik'],
                             key=lambda x: x['accuracy_mean'])
klasik_isim = klasik_sampiyon_kayit['model']  # 'XGBoost_top10'

# Disk'teki gerçek dosya yolu — tüm olası adları dene
klasik_olasi_dosyalar = [
    f'{OUTPUT_DIR}/sonuclar/klasik/klasik_{klasik_isim}_10f.json',
    f'{OUTPUT_DIR}/sonuclar/klasik/klasik_{klasik_isim}.json',
    f'{OUTPUT_DIR}/sonuclar/klasik/klasik_{klasik_isim.replace("_top10","")}_top10_10f.json',
]

klasik_yol = None
for olasi in klasik_olasi_dosyalar:
    if os.path.exists(olasi):
        klasik_yol = olasi
        break

# Eğer hâlâ bulunamazsa, glob ile ara
if klasik_yol is None:
    matches = glob.glob(f'{OUTPUT_DIR}/sonuclar/klasik/*XGBoost*top10*.json')
    if matches:
        klasik_yol = matches[0]

if klasik_yol is None:
    raise FileNotFoundError(f"Klasik şampiyon dosyası bulunamadı! "
                            f"Aranan: {klasik_olasi_dosyalar}")

print(f"   ✅ Klasik dosya  : {os.path.basename(klasik_yol)}")

# Klasik fold accuracy'lerini çek
with open(klasik_yol, 'r') as f:
    klasik_data = json.load(f)
klasik_fold_acc = [fs['accuracy'] for fs in klasik_data['fold_sonuclari']]

# Quantum şampiyon dosyası
quantum_yol = f'{OUTPUT_DIR}/sonuclar/quantum/quantum_Q_Hybrid_Q3Plus_ReUpAmp.json'
print(f"   ✅ Quantum dosya : {os.path.basename(quantum_yol)}")

with open(quantum_yol, 'r') as f:
    quantum_data = json.load(f)
quantum_fold_acc = [fs['accuracy'] for fs in quantum_data['fold_sonuclari']]

print(f"\n   Klasik şampiyon  : {klasik_isim}")
print(f"   Klasik fold acc  : {[f'{a:.4f}' for a in klasik_fold_acc]}")
print(f"   Quantum şampiyon : Q_Hybrid_Q3Plus_ReUpAmp")
print(f"   Quantum fold acc : {[f'{a:.4f}' for a in quantum_fold_acc]}")

# ── Adım 2: Wilcoxon Signed-Rank Test ──────────────────────
print(f"\n📌 ADIM 2: Wilcoxon signed-rank test uygulanıyor...")

if len(klasik_fold_acc) == 5 and len(quantum_fold_acc) == 5:
    klasik_arr  = np.array(klasik_fold_acc)
    quantum_arr = np.array(quantum_fold_acc)
    farklar     = klasik_arr - quantum_arr

    print(f"\n   Fold farkları (klasik - quantum):")
    for i, (k, q, d) in enumerate(zip(klasik_arr, quantum_arr, farklar), 1):
        print(f"     Fold {i}: {k:.4f} - {q:.4f} = {d:+.4f}")

    # Two-sided
    try:
        stat, p_value = stats.wilcoxon(klasik_arr, quantum_arr,
                                        alternative='two-sided')
        print(f"\n   📊 Wilcoxon two-sided test:")
        print(f"      Test statistic : {stat:.4f}")
        print(f"      p-value        : {p_value:.6f}")
        if p_value < 0.05:
            print(f"      ✅ p < 0.05: Fark istatistiksel olarak ANLAMLI")
        else:
            print(f"      ⚠️ p ≥ 0.05: Fark anlamlı DEĞİL "
                  f"(n=5'te min p=0.0625, beklenen davranış)")
    except Exception as e:
        print(f"      ⚠️ Hata: {e}")
        stat, p_value = None, None

    # One-sided (klasik > quantum)
    try:
        stat_g, p_value_g = stats.wilcoxon(klasik_arr, quantum_arr,
                                            alternative='greater')
        print(f"\n   📊 Wilcoxon one-sided (klasik > quantum) test:")
        print(f"      Test statistic : {stat_g:.4f}")
        print(f"      p-value        : {p_value_g:.6f}")
        if p_value_g < 0.05:
            print(f"      ✅ p < 0.05: Klasik şampiyonun ÜSTÜNLÜĞÜ anlamlı")
    except Exception as e:
        print(f"      ⚠️ Hata: {e}")
        stat_g, p_value_g = None, None

    # Cohen's d (paired)
    cohen_d = farklar.mean() / farklar.std(ddof=1) if farklar.std() > 0 else 0
    print(f"\n   📊 Effect Size (Cohen's d, paired):")
    print(f"      d = {cohen_d:.4f}")
    if abs(cohen_d) >= 0.8:
        etki_yorum = "BÜYÜK etki"
    elif abs(cohen_d) >= 0.5:
        etki_yorum = "ORTA etki"
    elif abs(cohen_d) >= 0.2:
        etki_yorum = "KÜÇÜK etki"
    else:
        etki_yorum = "İhmal edilebilir etki"
    print(f"      Yorum: {etki_yorum}")

    # Mean difference + 95% CI
    mean_fark = farklar.mean()
    se_fark = farklar.std(ddof=1) / np.sqrt(len(farklar))
    ci_low = mean_fark - 1.96 * se_fark
    ci_high = mean_fark + 1.96 * se_fark
    print(f"\n   📊 Ortalama Fark + %95 Güven Aralığı:")
    print(f"      Mean diff (klasik - quantum) = {mean_fark:.4f}")
    print(f"      95% CI = [{ci_low:.4f}, {ci_high:.4f}]")

    # ── Adım 3: Görsel ──────────────────────────────────────
    print(f"\n📊 GÖRSEL: Fold-bazlı Karşılaştırma")

    fig, ax = plt.subplots(figsize=(10, 6))
    x_pos = np.arange(1, 6)
    width = 0.35

    bars1 = ax.bar(x_pos - width/2, klasik_arr, width,
                   label=f'Klasik: {klasik_isim}',
                   color='#34495e', edgecolor='black', alpha=0.85)
    bars2 = ax.bar(x_pos + width/2, quantum_arr, width,
                   label='Quantum: Q-Hybrid-Q3-Plus',
                   color='#16a085', edgecolor='black', alpha=0.85)

    for bar, v in zip(bars1, klasik_arr):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.005,
                f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
    for bar, v in zip(bars2, quantum_arr):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.005,
                f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

    if p_value is not None and p_value_g is not None:
        title_extra = (f'\nWilcoxon two-sided p={p_value:.4f} | '
                       f'one-sided p={p_value_g:.4f} | '
                       f'Cohen\'s d={cohen_d:.2f}')
    else:
        title_extra = ''
    ax.set_title(f'Fold-bazlı Klasik vs Quantum Karşılaştırması{title_extra}\n'
                 'Obezite Veri Seti, 5-Fold Stratified CV',
                 fontweight='bold', fontsize=11)
    ax.set_xlabel('Fold', fontweight='bold', fontsize=11)
    ax.set_ylabel('Accuracy', fontweight='bold', fontsize=11)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'Fold {i}' for i in x_pos])
    ax.set_ylim([0.0, 1.05])
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    wilcoxon_yol = f'{GORSELLER}/wilcoxon_fold_karsilastirma.png'
    plt.savefig(wilcoxon_yol, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Kaydedildi: {wilcoxon_yol}")

    # ── Kayıt ──────────────────────────────────────────────
    wilcoxon_kayit = {
        "tarih"        : datetime.now().strftime('%d/%m/%Y %H:%M:%S'),
        "klasik_model" : klasik_isim,
        "quantum_model": "Q_Hybrid_Q3Plus_ReUpAmp",
        "klasik_fold_acc"  : [float(a) for a in klasik_fold_acc],
        "quantum_fold_acc" : [float(a) for a in quantum_fold_acc],
        "test": {
            "two_sided_stat" : float(stat) if stat is not None else None,
            "two_sided_p"    : float(p_value) if p_value is not None else None,
            "one_sided_stat" : float(stat_g) if stat_g is not None else None,
            "one_sided_p"    : float(p_value_g) if p_value_g is not None else None,
            "cohen_d"        : float(cohen_d),
            "cohen_d_yorum"  : etki_yorum,
            "mean_diff"      : float(mean_fark),
            "ci_95"          : [float(ci_low), float(ci_high)],
        }
    }

    wilcoxon_yol_json = f'{OUTPUT_DIR}/sonuclar/wilcoxon_test.json'
    with open(wilcoxon_yol_json, 'w', encoding='utf-8') as f:
        json.dump(wilcoxon_kayit, f, ensure_ascii=False, indent=4, default=str)
    print(f"\n💾 JSON kaydedildi: {wilcoxon_yol_json}")

    # ── Özet ───────────────────────────────────────────────
    print(f"\n{'='*65}")
    print(f"{'='*65}")
    print(f"   Two-sided p   : {p_value:.6f}")
    print(f"   One-sided p   : {p_value_g:.6f}")
    print(f"   Cohen's d     : {cohen_d:.4f} ({etki_yorum})")
    print(f"   Mean diff     : {mean_fark:.4f}")
    print(f"   95% CI        : [{ci_low:.4f}, {ci_high:.4f}]")
else:
    print(f"   ⚠️ Hata: Fold sayıları beklenmedik")
    print(f"      Klasik: {len(klasik_fold_acc)}, Quantum: {len(quantum_fold_acc)}")


In [ ]:
# Tüm sonuçların master birleştirme dosyası, çıktı envanteri,
# makale-hazır rakam tablosu

print("=" * 65)
print("🏁 FİNAL KONSOLİDASYON — OBEZİTE FAZI MAKALE-HAZIR ÖZET")
print("=" * 65)

import os, glob

# ── ADIM 1: Tüm çıktı dosyalarının envanterini çıkar ───────
print(f"\n📌 ADIM 1: Drive manifest oluşturuluyor...")

manifest = {
    "tarih": datetime.now().strftime('%d/%m/%Y %H:%M:%S'),
    "calisma": "QML_Obesity_ISADES2026",
    "veri_seti": "Palechor 2019 Obesity Estimation",
    "n_orneklem_train": int(len(QUANTUM_VERILER['y_train'])),
    "n_orneklem_test": int(len(QUANTUM_VERILER['y_test'])),
    "n_sinif": int(N_CLASSES),
    "klasor_yapisi": {}
}

# Klasör tarama
ana_klasorler = ['sonuclar', 'sonuclar/klasik', 'sonuclar/quantum',
                 'gorseller', 'modeller', 'checkpoints', 'veri']
for klasor in ana_klasorler:
    full_path = f'{OUTPUT_DIR}/{klasor}'
    if os.path.exists(full_path):
        dosyalar = sorted([os.path.basename(f)
                           for f in glob.glob(f'{full_path}/*')
                           if os.path.isfile(f)])
        manifest['klasor_yapisi'][klasor] = {
            'n_dosya': len(dosyalar),
            'dosyalar': dosyalar
        }
        print(f"   {klasor:<25}: {len(dosyalar)} dosya")

# ── ADIM 2: Tüm Sonuçları Birleştir ─────────────────────────
print(f"\n📌 ADIM 2: Master sonuç birleştiriliyor...")

klasik_sampiyon = max(TUM_SONUCLAR['klasik'], key=lambda x: x['accuracy_mean'])
quantum_sampiyon = max(TUM_SONUCLAR['quantum'], key=lambda x: x['accuracy_mean'])

# Re-upload ablation
reupload_ablation = {}
for s in TUM_SONUCLAR['quantum']:
    if 'ReUpload' in s['model'] and 'block' in s['model']:
        if '1block' in s['model']:
            reupload_ablation['1_block'] = {'acc_mean': s['accuracy_mean'],
                                             'acc_std': s['accuracy_std']}
        elif '2block' in s['model']:
            reupload_ablation['2_block'] = {'acc_mean': s['accuracy_mean'],
                                             'acc_std': s['accuracy_std']}
        elif '3block' in s['model']:
            reupload_ablation['3_block'] = {'acc_mean': s['accuracy_mean'],
                                             'acc_std': s['accuracy_std']}

# Encoding aile özeti
encoding_aile = {
    'pure_angle': max([s for s in TUM_SONUCLAR['quantum']
                       if 'Angle' in s['model'] and 'Q3' not in s['model']],
                      key=lambda x: x['accuracy_mean'], default=None),
    'pure_amplitude': max([s for s in TUM_SONUCLAR['quantum']
                           if 'Amplitude' in s['model']
                           and 'Q3' not in s['model']],
                          key=lambda x: x['accuracy_mean'], default=None),
    'reupload_pure': max([s for s in TUM_SONUCLAR['quantum']
                          if 'ReUpload' in s['model']
                          and 'block' in s['model']],
                         key=lambda x: x['accuracy_mean'], default=None),
    'hybrid_q3': max([s for s in TUM_SONUCLAR['quantum']
                      if 'Q3' in s['model'] and 'Plus' not in s['model']],
                     key=lambda x: x['accuracy_mean'], default=None),
    'hybrid_q3plus': max([s for s in TUM_SONUCLAR['quantum']
                          if 'Q3Plus' in s['model']],
                         key=lambda x: x['accuracy_mean'], default=None),
}

# Wilcoxon dosyasından
wilcoxon_yol_json = f'{OUTPUT_DIR}/sonuclar/wilcoxon_test.json'
if os.path.exists(wilcoxon_yol_json):
    with open(wilcoxon_yol_json, 'r') as f:
        wilcoxon_data = json.load(f)
else:
    wilcoxon_data = None

# SHAP dosyasından
shap_yol_json = f'{OUTPUT_DIR}/sonuclar/shap_xgboost.json'
if os.path.exists(shap_yol_json):
    with open(shap_yol_json, 'r') as f:
        shap_data = json.load(f)
else:
    shap_data = None

# Master özet
master_ozet = {
    "tarih": datetime.now().strftime('%d/%m/%Y %H:%M:%S'),
    "calisma": "QML for Obesity Multiclass Classification — ISADES 2026",

    "veri_bilgisi": {
        "kaynak": "Palechor 2019 (UCI ML Repository)",
        "n_orneklem_total": int(len(QUANTUM_VERILER['y_train']) +
                                len(QUANTUM_VERILER['y_test'])),
        "n_orneklem_train": int(len(QUANTUM_VERILER['y_train'])),
        "n_orneklem_test": int(len(QUANTUM_VERILER['y_test'])),
        "n_ozellik_tum": 16,
        "n_ozellik_top10": 10,
        "n_sinif": int(N_CLASSES),
        "siniflar": list(KLASIK_VERILER['label_encoder'].classes_)
    },

    "klasik_sonuclar": {
        "n_model": len(TUM_SONUCLAR['klasik']),
        "sampiyon": {
            "model": klasik_sampiyon['model'],
            "cv_acc_mean": float(klasik_sampiyon['accuracy_mean']),
            "cv_acc_std": float(klasik_sampiyon['accuracy_std']),
            "cv_f1_macro": float(klasik_sampiyon['f1_macro_mean']),
            "cv_mcc": float(klasik_sampiyon['mcc_mean']),
            "cv_auc_macro": float(klasik_sampiyon['auc_macro_mean']),
        },
        "tum_modeller_siralama": sorted(
            [{"model": s['model'],
              "cv_acc": float(s['accuracy_mean']),
              "cv_acc_std": float(s['accuracy_std']),
              "cv_auc": float(s.get('auc_macro_mean', 0))}
             for s in TUM_SONUCLAR['klasik']],
            key=lambda x: x['cv_acc'], reverse=True
        )
    },

    "quantum_sonuclar": {
        "n_model": len(TUM_SONUCLAR['quantum']),
        "sampiyon": {
            "model": quantum_sampiyon['model'],
            "cv_acc_mean": float(quantum_sampiyon['accuracy_mean']),
            "cv_acc_std": float(quantum_sampiyon['accuracy_std']),
            "cv_f1_macro": float(quantum_sampiyon['f1_macro_mean']),
            "cv_mcc": float(quantum_sampiyon['mcc_mean']),
            "cv_auc_macro": float(quantum_sampiyon['auc_macro_mean']),
        },
        "tum_modeller_siralama": sorted(
            [{"model": s['model'],
              "cv_acc": float(s['accuracy_mean']),
              "cv_acc_std": float(s['accuracy_std']),
              "cv_auc": float(s.get('auc_macro_mean', 0))}
             for s in TUM_SONUCLAR['quantum']],
            key=lambda x: x['cv_acc'], reverse=True
        ),
        "reupload_ablation": reupload_ablation,
        "encoding_aile_ozet": {
            aile: ({"model": v['model'], "cv_acc": float(v['accuracy_mean'])}
                   if v else None)
            for aile, v in encoding_aile.items()
        }
    },

    "karsilastirma": {
        "klasik_quantum_acc_gap_puan": float(
            (klasik_sampiyon['accuracy_mean'] -
             quantum_sampiyon['accuracy_mean']) * 100
        ),
        "klasik_quantum_auc_gap": float(
            klasik_sampiyon['auc_macro_mean'] -
            quantum_sampiyon['auc_macro_mean']
        ),
        "wilcoxon_test": (wilcoxon_data['test'] if wilcoxon_data else None)
    },

    "shap_analizi": (shap_data if shap_data else None),

    "metodoloji_ozet": {
        "cv_strategy": "5-fold StratifiedKFold (random_state=42)",
        "scaler": "StandardScaler (sayısal) + OneHotEncoder (kategorik)",
        "klasik_tuning": "GridSearchCV, 5-fold inner CV",
        "quantum_egitim": {
            "framework": "PennyLane 0.32+ + PyTorch",
            "device": "default.qubit",
            "interface": "torch",
            "diff_method": "backprop",
            "optimizer": "Adam (lr=0.05) + CosineAnnealingLR",
            "loss": "CrossEntropyLoss (class-weighted, balanced)",
            "epochs_max": 100,
            "early_stopping": "patience=15",
            "batch_size": 32
        },
        "istatistik": "Paired Wilcoxon signed-rank + Cohen's d",
        "yorumlanabilirlik": "SHAP TreeExplainer (XGBoost)"
    }
}

# Master JSON kaydet
master_yol = f'{OUTPUT_DIR}/sonuclar/MASTER_OZET_OBEZITE.json'
with open(master_yol, 'w', encoding='utf-8') as f:
    json.dump(master_ozet, f, ensure_ascii=False, indent=4, default=str)
print(f"   💾 MASTER_OZET_OBEZITE.json kaydedildi")

# Manifest kaydet
manifest_yol = f'{OUTPUT_DIR}/sonuclar/MANIFEST.json'
with open(manifest_yol, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=4, default=str)
print(f"   💾 MANIFEST.json kaydedildi")

# ── ADIM 3: Makale-Hazır Rakam Tablosu ─────────────────────
print(f"\n{'='*65}")
print(f"📋 MAKALE-HAZIR RAKAM TABLOSU (KOPYALA-YAPISTIR)")
print(f"{'='*65}")

print(f"\n### Tablo: Obezite Veri Seti — Tüm Modeller")
print(f"\n{'Model':<35} {'CV Acc (Mean±Std)':<22} {'AUC':<10} {'F1':<10}")
print(f"{'-'*35} {'-'*22} {'-'*10} {'-'*10}")

# Önce klasikler
for s in sorted(TUM_SONUCLAR['klasik'],
                 key=lambda x: x['accuracy_mean'], reverse=True):
    if 'top10' in s['model']:
        print(f"{s['model']:<35} "
              f"{s['accuracy_mean']:.4f}±{s['accuracy_std']:.4f}     "
              f"{s.get('auc_macro_mean', 0):.4f}    "
              f"{s.get('f1_macro_mean', 0):.4f}")

print()

# Quantum'lar
for s in sorted(TUM_SONUCLAR['quantum'],
                 key=lambda x: x['accuracy_mean'], reverse=True):
    print(f"{s['model']:<35} "
          f"{s['accuracy_mean']:.4f}±{s['accuracy_std']:.4f}     "
          f"{s.get('auc_macro_mean', 0):.4f}    "
          f"{s.get('f1_macro_mean', 0):.4f}")

# ── ADIM 4: Çapraz-Bağlam Karşılaştırma (WBCD vs Obezite) ──
print(f"\n{'='*65}")
print(f"📊 ÇAPRAZ-BAĞLAM KARŞILAŞTIRMASI (Makale Anahtar Tablosu)")
print(f"{'='*65}")

print(f"\n### Tablo: WBCD vs Obezite — Çapraz Karşılaştırma")
print(f"\n{'Boyut':<28} {'WBCD':<25} {'Obezite':<25}")
print(f"{'-'*28} {'-'*25} {'-'*25}")
print(f"{'Sınıf sayısı':<28} {'2 (binary)':<25} {'7 (multiclass)':<25}")
print(f"{'Veri kaynağı':<28} {'%100 gerçek':<25} {'%77 sentetik (SMOTE)':<25}")
print(f"{'Klasik şampiyon':<28} "
      f"{'SVM-RBF: %98.02':<25} "
      f"{'XGBoost: ' + f'%{klasik_sampiyon[chr(34)+chr(97)+chr(99)+chr(99)+chr(117)+chr(114)+chr(97)+chr(99)+chr(121)+chr(95)+chr(109)+chr(101)+chr(97)+chr(110)+chr(34)] * 100:.2f}':<25}")

klasik_acc = klasik_sampiyon['accuracy_mean'] * 100
quantum_acc = quantum_sampiyon['accuracy_mean'] * 100
gap = (klasik_sampiyon['accuracy_mean'] -
       quantum_sampiyon['accuracy_mean']) * 100

print(f"\n   Düzeltilmiş satırlar:")
print(f"   Klasik şampiyon  : WBCD SVM-RBF %98.02  | Obezite XGBoost %{klasik_acc:.2f}")
print(f"   Quantum şampiyon : WBCD ReUpload-3blk %92.97 | "
      f"Obezite Q3-Plus %{quantum_acc:.2f}")
print(f"   Acc gap          : WBCD ~5 puan        | Obezite ~{gap:.1f} puan")
print(f"   AUC gap          : WBCD 0.005          | "
      f"Obezite {(klasik_sampiyon['auc_macro_mean']-quantum_sampiyon['auc_macro_mean']):.3f}")

# ── ADIM 5: Final Çıktı ────────────────────────────────────
print(f"\n{'='*65}")
print(f"{'='*65}")

print(f"\n📌 ÖZET RAKAMLAR:")
print(f"   Toplam klasik model deneyi  : {len(TUM_SONUCLAR['klasik'])}")
print(f"   Toplam quantum model deneyi : {len(TUM_SONUCLAR['quantum'])}")
print(f"   Klasik şampiyon            : {klasik_sampiyon['model']} "
      f"({klasik_sampiyon['accuracy_mean']*100:.2f}%)")
print(f"   Quantum şampiyon           : {quantum_sampiyon['model']} "
      f"({quantum_sampiyon['accuracy_mean']*100:.2f}%)")
print(f"   Acc gap (klasik-quantum)   : {gap:.2f} puan")
print(f"   Wilcoxon p (one-sided)     : "
      f"{wilcoxon_data['test']['one_sided_p']:.4f} (anlamlı)" if wilcoxon_data else "-")
print(f"   Cohen's d                  : "
      f"{wilcoxon_data['test']['cohen_d']:.2f} (BÜYÜK etki)" if wilcoxon_data else "-")

print(f"\n📂 DRIVE'da TOPLAM:")
toplam_dosya = sum(v['n_dosya'] for v in manifest['klasor_yapisi'].values())
print(f"   {toplam_dosya} dosya, {len(manifest['klasor_yapisi'])} klasör")
